# FoodVision AI — Classificazione di immagini di cibo per GourmetAI Inc.

> **Cliente**: GourmetAI Inc. — food-tech

> **Algoritmo principale**: Transfer Learning con EfficientNet-B0 (`timm` + PyTorch)

> **Dataset**: 14 categorie di piatti, ~14.000 immagini fotografiche reali

## Contesto

GourmetAI Inc. vuole automatizzare il riconoscimento dei piatti fotografati dagli utenti delle sue app (ricette, food-delivery, diari alimentari). Oggi la categorizzazione è manuale o affidata a tag inseriti dagli utenti stessi, il che la rende lenta e incoerente: due persone possono taggare la stessa foto di patate al forno in modi diversi, o non taggarla affatto.

Un classificatore automatico che riconosce il piatto direttamente dalla foto permette di:

- **etichettare i contenuti in automatico**, senza dipendere dall'utente;
- **velocizzare le ricerche e i suggerimenti** nell'app (es. "mostrami altre ricette come questa");
- **arricchire i dati interni** di GourmetAI con informazioni strutturate su cosa viene effettivamente fotografato/consumato dagli utenti.

## Il dataset

Il dataset fornito contiene **14 classi di piatti**, organizzate già in tre split (train/val/test):

`Baked Potato`, `Crispy Chicken`, `Donut`, `Fries`, `Hot Dog`, `Sandwich`, `Taco`, `Taquito`, `apple_pie`, `cheesecake`, `chicken_curry`, `ice_cream`, `omelette`, `sushi`

Sono fotografie reali (foto da ricettario/blog e foto "da utente"), non immagini sintetiche o renderizzate: è quindi un caso di **classificazione multiclasse di immagini fotografiche naturali**.

## Obiettivo del progetto

Sviluppare un classificatore basato su **Deep Learning** e **transfer learning** che raggiunga buone prestazioni su tutte e 14 le classi (non solo sulle più "facili"), seguendo un percorso sperimentale che parte da una baseline semplice e aggiunge progressivamente tecniche di augmentation, fine-tuning e regolarizzazione — misurando ad ogni passo l'effetto della singola modifica.

## Metodologia adottata

Usiamo modelli **EfficientNet preaddestrati su ImageNet** (B0 e, nell'ultimo esperimento, B2), con una testa di classificazione adattata alle 14 classi del dataset.

Il percorso comprende EDA e pulizia del dataset, controllo degli split forniti, augmentation solo sul training, confronti controllati su validation e valutazione finale sul test del solo modello selezionato. La metrica primaria è il **F1 macro**; accuracy, top-3 accuracy e metriche per classe completano il report.

Gli esperimenti sono organizzati in due fasi:

- **sette esperimenti controllati** con EfficientNet-B0, che modificano un fattore alla volta tra augmentation, strategia di fine-tuning, Label Smoothing e Mixup;
- **due esperimenti di approfondimento** a partire dalla configurazione migliore dei sette: l'esperimento 8 aumenta la risoluzione delle immagini da 224 a 288 pixel, l'esperimento 9 sostituisce EfficientNet-B0 con EfficientNet-B2 mantenendo la risoluzione di 288 pixel.

Tutti gli esperimenti usano lo stesso seed e lo stesso budget massimo di 25 epoche, con early stopping sul F1 macro di validation per rendere i confronti più coerenti.

1. Setup e download riutilizzabile del dataset
2. Librerie, seed e device
3. EDA e verifica delle classi nei tre split
4. Configurazione centralizzata
5. `ExperimentRunner`: dati, training e checkpoint
6. `Evaluator`: metriche, curve e matrice di confusione
7. Configurazione degli esperimenti (sette controllati e due di approfondimento)
8. Esperimento di approfondimento: risoluzione più alta
9. Esperimento di approfondimento: modello più grande (EfficientNet-B2)
10. Confronto e selezione automatica
11. Test finale
12. Analisi degli errori dalle predizioni già calcolate
13. Conclusioni


## 1. Setup dell'ambiente e download del dataset

Il notebook è pensato per essere eseguito su **Google Colab con runtime GPU**
(Runtime → Cambia tipo di runtime → GPU). `timm` non è preinstallato su Colab, va quindi
installato esplicitamente.

In [ ]:
!pip install -q timm

In [ ]:
# ============================================================
# CELLA 1 — Download e preparazione del dataset
# ============================================================
# Il dataset è ospitato su S3 e viene scaricato solo se non è già presente.
# Il dataset food è un archivio .zip: viene usato unzip per estrarlo.
# ============================================================

import os
import zipfile

DATASET_URL = "https://proai-datasets.s3.eu-west-3.amazonaws.com/dataset_food_classification.zip"
DATASET_ARCHIVE = "dataset_food_classification.zip"
DATASET_FOLDER = "dataset"
DATASET_ROOT = "datasets/dataset_food_classification"

# Controllo la destinazione finale. Se non presente Avvio del dowload
if not os.path.isdir(DATASET_ROOT) and not os.path.isdir(DATASET_FOLDER):
    if not zipfile.is_zipfile(DATASET_ARCHIVE):
        print("Dataset non trovato in locale. Avvio download...")
        # !wget scarica il file dall'URL e lo salva come DATASET_ARCHIVE
        !wget -O {DATASET_ARCHIVE} {DATASET_URL}
    if not zipfile.is_zipfile(DATASET_ARCHIVE):
        raise RuntimeError("Download non riuscito o archivio ZIP incompleto.")
    # unzip: estrazione del dataset
    !unzip -q -o {DATASET_ARCHIVE}
else:
    print("Dataset già presente: non serve scaricarlo nuovamente.")



In [ ]:
# Spostiamo la cartella del dataset in "datasets/" per mantenere il progetto ordinato.
# Il controllo sulla destinazione evita di ripetere lo spostamento a ogni esecuzione.
if not os.path.isdir(DATASET_ROOT):
    !mkdir -p datasets
    !mv {DATASET_FOLDER} {DATASET_ROOT}

# Gli split sono già forniti: controlliamo le cartelle, senza valutare il modello sul test.
for split in ("train", "val", "test"):
    if not os.path.isdir(os.path.join(DATASET_ROOT, split)):
        raise ValueError(f"Split mancante: {split}")
print("Dataset pronto:", DATASET_ROOT)


In [ ]:
# ============================================================
# CELLA 1B — Cartella persistente per i checkpoint (Google Drive)
# ============================================================
# Il disco del runtime Colab viene cancellato quando la sessione termina o si disconnette.
# Vengono salvati quindi i checkpoint degli esperimenti su Google Drive: se la sessione cade,
# gli esperimenti già completati vengono ricaricati dal checkpoint.
# Fuori da Colab (o con USE_DRIVE = False) i checkpoint restano in una cartella locale.
# ============================================================

USE_DRIVE = True
CHECKPOINT_DIR = "checkpoints"

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_DIR = "/content/drive/MyDrive/FoodVision_checkpoints"
    except ImportError:
        print("Google Colab non rilevato: i checkpoint vengono salvati in locale.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Cartella dei checkpoint:", CHECKPOINT_DIR)

## 2. Import delle librerie, seed e device

In [ ]:

# ============================================================
# CELLA 2 — Import delle librerie
# ============================================================

import copy          # copy.deepcopy() per clonare configurazioni e checkpoint
import os            # operazioni su file e directory
import random        # seed del generatore Python standard
import time          # misura la durata di ogni epoca

from collections import Counter
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Con nove esperimenti, ciascuno con più grafici (loss/F1, matrice di confusione, coppie
# confuse), il numero di figure aperte in una sessione supera facilmente la soglia di
# default di matplotlib: disattiviamo il relativo warning, non indica un problema reale.
plt.rcParams["figure.max_open_warning"] = 0

# PyTorch: framework principale per costruzione e training del modello
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# ImageFolder: carica automaticamente le immagini organizzate per sottocartella di classe
from torchvision.datasets import ImageFolder

# timm (PyTorch Image Models): modelli pretrained e utility per il transfer learning
import timm
from timm.data import create_transform, resolve_data_config
from timm.data.mixup import Mixup
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy

# sklearn: metriche di valutazione
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    top_k_accuracy_score,
)


In [ ]:

# ============================================================
# CELLA 3 — Seed e selezione del device
# ============================================================

def set_seed(seed: int = 42) -> None:
    """
    Imposta i seed su tutti i generatori di numeri casuali per garantire
    la riproducibilità degli esperimenti.

    PyTorch usa più sorgenti di casualità indipendenti:
    - random: operazioni Python standard
    - numpy: operazioni di array e sklearn (es. train_test_split)
    - torch (CPU): inizializzazione pesi e dropout
    - torch (GPU): operazioni su CUDA
    - cudnn.deterministic: forza CUDA a usare algoritmi deterministici
    - cudnn.benchmark=False: disabilita la selezione automatica algoritmo (non deterministica)

    Trade-off: deterministic=True rallenta leggermente il training ma garantisce
    che rieseguire il notebook due volte produca esattamente gli stessi risultati.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Forza CUDA a usare algoritmi deterministici (a costo di un training ~10-15% più lento)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


SEED = 42
set_seed(SEED)

# Selezione automatica del device: GPU se disponibile (Colab), altrimenti CPU.
# Una CNN come EfficientNet-B0 su ~9.000 immagini di training è allenabile anche
# su CPU per una verifica rapida, ma il training completo va eseguito su GPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device in uso:", device)

if device.type == "cuda":
    print("GPU rilevata:", torch.cuda.get_device_name(0))
else:
    print("ATTENZIONE: nessuna GPU rilevata. Su Colab: Runtime -> Cambia tipo di runtime -> GPU.")


## 3. Esplorazione del dataset (EDA)

Prima di costruire e addestrare il modello viene analizzato il dataset per comprenderne la struttura e verificare la qualità dei dati disponibili.

In particolare, l'analisi comprende:

- il controllo delle **14 classi** presenti nei tre split (`train`, `validation` e `test`);
- il conteggio delle immagini per classe e per split, per verificare la **distribuzione delle classi**;
- un controllo dell'**integrità dei file**, delle dimensioni e del formato delle immagini;
- la ricerca di **duplicati esatti**, sia all'interno dello stesso split sia tra split differenti;
- l'individuazione di eventuali immagini duplicate associate a **etichette diverse**;
- un controllo visivo di un campione di immagini per valutare qualitativamente la coerenza delle classi.

Questi controlli permettono di individuare eventuali problemi nel dataset **prima dell'addestramento** e di definire, quando necessario, le operazioni di pulizia da applicare.

I controlli automatici possono verificare la struttura dei file e individuare duplicati esatti, ma **non possono garantire che tutte le immagini siano etichettate correttamente**. Per questo motivo viene affiancato anche un controllo visivo campionario.


In [ ]:
# ============================================================
# CELLA 4 — Conteggio immagini per classe e per split
# ============================================================
# Le cartelle di classe hanno nomi disomogenei: 
# - alcune cartelle usano parole separate da spazi e iniziali maiuscole
#   (es. "Baked Potato", "Crispy Chicken", "Hot Dog");
# - altre usano lettere minuscole e underscore
#   (es. "apple_pie", "chicken_curry", "ice_cream").
# Per la pipeline (ImageFolder) questo non cambia nulla: ogni cartella resta una
# classe valida. Definiamo però una mappa verso un nome "da mostrare" più pulito,
# usata solo nei grafici e nei report, senza toccare le cartelle su disco.

# Creo un dataset ImageFolder per ogni split
split_datasets = {split: ImageFolder(os.path.join(DATASET_ROOT, split))
                  for split in ("train", "val", "test")}
# Recupero i nomi delle classi dal training set
RAW_CLASSES = split_datasets["train"].classes
# Controllo che validation e test abbiano esattamente le stesse classi e gli stessi indici
if any(ds.class_to_idx != split_datasets["train"].class_to_idx for ds in split_datasets.values()):
    raise ValueError("Le classi o le etichette numeriche differiscono tra gli split")

DISPLAY_NAME = {
    "Baked Potato": "Baked Potato",
    "Crispy Chicken": "Crispy Chicken",
    "Donut": "Donut",
    "Fries": "Fries",
    "Hot Dog": "Hot Dog",
    "Sandwich": "Sandwich",
    "Taco": "Taco",
    "Taquito": "Taquito",
    "apple_pie": "Apple Pie",
    "cheesecake": "Cheesecake",
    "chicken_curry": "Chicken Curry",
    "ice_cream": "Ice Cream",
    "omelette": "Omelette",
    "sushi": "Sushi",
}

# ds.targets è un attributo di ImageFolder che contiene l'etichetta numerica di ogni 
# immagine del dataset.
counts = {split: Counter(ds.classes[target] for target in ds.targets)
          for split, ds in split_datasets.items()}
print(counts)
print("----------------------------------------------------------------------------")

# loc[RAW_CLASSES] = righe/indice → nomi delle classi
dist_df = pd.DataFrame(counts).loc[RAW_CLASSES]
dist_df.index = [DISPLAY_NAME.get(c, c) for c in RAW_CLASSES]
dist_df["totale"] = dist_df.sum(axis=1)
print(dist_df)
print("\nTotale immagini nel dataset:", dist_df["totale"].sum())


In [ ]:
# Grafico a barre della distribuzione delle classi nel training set.
# È una funzione per poterla richiamare dopo la pulizia della cella 5.5
# e confrontare visivamente la distribuzione prima e dopo.
def plot_train_distribution(counts_df, title):
    labels = [DISPLAY_NAME.get(c, c) for c in counts_df.index]
    plt.figure(figsize=(11, 4))
    plt.bar(labels, counts_df["train"], color="#4c72b0")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Numero di immagini (train)")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_train_distribution(dist_df, "Distribuzione delle 14 classi nel training set")

### Analisi della distribuzione delle classi

Il dataset contiene **14 classi** e risulta perfettamente bilanciato: ogni classe contiene **1.000 immagini**, suddivise nello stesso modo tra i tre split:

- **640 immagini** nel training set;
- **160 immagini** nel validation set;
- **200 immagini** nel test set.

Ogni classe mantiene quindi la stessa proporzione **64% train, 16% validation e 20% test**, per un totale complessivo di **14.000 immagini**.

Il grafico conferma l'assenza di sbilanciamento nel training set: tutte le barre hanno infatti la stessa altezza. Non è quindi necessario applicare tecniche di riequilibrio delle classi, come oversampling, undersampling o pesi differenti nella funzione di loss.

In [ ]:

# ============================================================
# CELLA 5 — Griglia di immagini di esempio, una per classe
# ============================================================

fig, axes = plt.subplots(2, 7, figsize=(18, 6))
for ax, cls in zip(axes.flat, RAW_CLASSES):
    class_dir = os.path.join(DATASET_ROOT, "train", cls)
    example_file = sorted(os.listdir(class_dir))[0]
    img = Image.open(os.path.join(class_dir, example_file))
    ax.imshow(img)
    ax.set_title(DISPLAY_NAME[cls], fontsize=10)
    ax.axis("off")
plt.suptitle("Un esempio per ciascuna delle 14 classi", y=1.02)
plt.tight_layout()
plt.show()


### Analisi visiva del dataset

La griglia mostra un'immagine di esempio per ciascuna delle **14 classi** e permette una prima verifica qualitativa del dataset.

Si osserva una discreta **variabilità visiva tra le immagini**: cambiano l'inquadratura, lo sfondo, l'illuminazione, le dimensioni del cibo nell'immagine e il contesto in cui viene fotografato. In alcuni casi l'alimento occupa gran parte dell'immagine, mentre in altri sono presenti anche piatti, bevande, persone o altri elementi di sfondo.

Alcune classi possono inoltre presentare **caratteristiche visive simili**: ad esempio alimenti con forme, colori o ingredienti comuni potrebbero rendere la classificazione meno immediata.

Questa prima esplorazione suggerisce quindi che il modello dovrà imparare a riconoscere le caratteristiche distintive del cibo anche in presenza di **variazioni di contesto e aspetto**.

> La griglia contiene però una sola immagine per classe: permette una prima osservazione qualitativa, ma non è sufficiente per descrivere tutta la variabilità presente all'interno di ciascuna classe.


In [ ]:
# ============================================================
# CELLA 5.1 — Controllo di qualità delle immagini
# ============================================================
# Vengono controllate tutte le immagini di train, validation e test per:
# 1. individuare eventuali file corrotti;
# 2. verificarne dimensioni, formato e canali;
# 3. trovare duplicati esatti tramite SHA-256;
# 4. verificare che la stessa immagine non compaia in split diversi
#    (possibile data leakage);
# 5. verificare che immagini identiche non abbiano etichette diverse.
# Per trovare i duplicati utilizziamo SHA-256:
# ogni file viene trasformato in una "impronta digitale" (hash).
# File identici byte per byte producono lo stesso hash.

import hashlib

def inspect_dataset(split_datasets):
    # immagini valide
    image_records = []
    # immagini corrotte
    corrupted_images = []
    # Analizzo separatamente train, validation e test
    for split, dataset in split_datasets.items():
        for path, label in dataset.samples:
            # Creazione di un record con le informazioni di base dell'immagine
            record = {"split": split, "class_name": dataset.classes[label], "path": path}
            try:
                # --- 3. CALCOLO DELL'HASH SHA-256
                with open(path, "rb") as file:
                    digest = hashlib.sha256()
                    for chunk in iter(lambda: file.read(1024 * 1024), b""):
                        digest.update(chunk)
                # --- 1. CONTROLLO DELL'INTEGRITÀ DEL FILE
                # verify() controlla che la struttura del file immagine
                # sia valida e permette di individuare file corrotti.
                with Image.open(path) as img:
                    img.verify()
                # --- 2. LETTURA COMPLETA DELL'IMMAGINE -
                # VERIFICA DIMENSIONI, FORMATO E CANALI
                # load() forza la decodifica dei pixel: in questo modo
                # controlliamo che l'immagine sia realmente utilizzabile
                # e non soltanto formalmente valida.
                with Image.open(path) as img:
                    img.load()
                    width, height = img.size
                    # Aggiungiamo al record tutte le informazioni tecniche
                    record.update(
                        width=width, 
                        height=height, 
                        aspect_ratio=width / height,
                        mode=img.mode, 
                        channels=len(img.getbands()), 
                        format=img.format,
                        sha256=digest.hexdigest()
                      )
                # Se non si sono verificati errori,
                # vengono salvate le informazioni dell'immagine
                image_records.append(record)
            except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as error:
                corrupted_images.append({**record, "error": str(error)})
        print(f"Scansione completata: {split} ({len(dataset.samples)} file)")

    image_info = pd.DataFrame(
        image_records, 
        columns=[
            "split", "class_name", "path", "width", "height", 
            "aspect_ratio", "mode", "channels", "format", "sha256"
          ]
        )
    # 1. FILE CORROTTI
    corrupted_images = pd.DataFrame(corrupted_images, columns=["split", "class_name", "path", "error"])
    # 3. IMMAGINI DUPLICATE TRAMITE HASH
    duplicate_images = image_info[image_info.duplicated("sha256", keep=False)].copy()
    # 4. DUPLICATI TRA SPLIT DIVERSI
    duplicate_images["across_splits"] = duplicate_images.groupby("sha256")["split"].transform("nunique") > 1
    # 5. DUPLICATI CON ETICHETTE DIVERSE
    # Esempio:
    # - stessa immagine → "Taco"
    # - stessa immagine → "Hot Dog"
    duplicate_images["conflicting_labels"] = duplicate_images.groupby("sha256")["class_name"].transform("nunique") > 1
    return image_info, corrupted_images, duplicate_images

In [ ]:
image_info, corrupted_images, duplicate_images = inspect_dataset(split_datasets)
print("File leggibili:", len(image_info), "| File non leggibili:", len(corrupted_images))
print("File coinvolti in gruppi di duplicati esatti:", len(duplicate_images))
print("File duplicati tra split:", int(duplicate_images["across_splits"].sum()))
print("File duplicati con etichette diverse:", int(duplicate_images["conflicting_labels"].sum()))

In [ ]:
# ============================================================
# SALVATAGGIO DEI REPORT DEL CONTROLLO
# ============================================================
# I DataFrame creati durante il controllo possono contenere molte righe, 
# li salvo quindi come file CSV, in modo da poter analizzare
# tutti i risultati.

# Cazione della cartella che conterrà i report.
os.makedirs("dataset_audit", exist_ok=True)
# Salvataggio di tutte le informazioni tecniche di tutte le immagini:
# split, classe, dimensioni, formato, canali, hash...
image_info.to_csv("dataset_audit/image_info.csv", index=False)
# Salvataggio degli eventuali file corrotti/non leggibili.
corrupted_images.to_csv("dataset_audit/corrupted_images.csv", index=False)
# Salvataggio degli eventuali duplicati esatti.
duplicate_images.to_csv("dataset_audit/duplicate_images.csv", index=False)

# Se sono stati trovati duplicati, vengono mostrati nel notebook
# i primi 20 casi per poterli esaminare rapidamente.
if not duplicate_images.empty:
    print(duplicate_images[["split", "class_name", "path", "across_splits", "conflicting_labels"]].head(20).to_string(index=False))
    print("Esaminare i duplicati prima di interpretare le metriche: nessun file è stato rimosso.")

# Se sono stati trovati immagini corrotte, vengono mostrati nel notebook
# i primi 20 casi per poterli esaminare rapidamente.
if not corrupted_images.empty:
    print(corrupted_images.head(20).to_string(index=False))
    raise ValueError("File non leggibili: controllare il report, correggere i dati e rieseguire l'EDA prima del training.")


### Controllo di integrità e duplicati

La scansione è stata eseguita su tutte le **14.000 immagini** del dataset. Non sono stati rilevati file corrotti o non leggibili.

Il controllo tramite hash SHA-256 ha però individuato **371 file coinvolti in gruppi di duplicati esatti**. In particolare, **199 file duplicati compaiono in split differenti** e **5 file duplicati risultano associati a etichette diverse**.

La presenza di duplicati tra train, validation e test richiede particolare attenzione perché può causare **data leakage**: il modello potrebbe essere valutato su immagini identiche a quelle già viste durante il training, producendo metriche eccessivamente ottimistiche.

I duplicati con etichette differenti devono invece essere analizzati separatamente, perché potrebbero indicare possibili errori o ambiguità nell'etichettatura.

Per questo motivo i duplicati non vengono rimossi automaticamente, ma vengono prima identificati e analizzati per stabilire come gestirli senza alterare impropriamente il dataset.

In [ ]:
# ============================================================
# CELLA 5.2 — Analisi dei duplicati e proposta di gestione
# ============================================================
# OBIETTIVO:
# creare un piano per gestire le immagini duplicate individuate
# tramite SHA-256 nella cella precedente.
#
# Regole utilizzate:
# 1. Se più file hanno lo stesso SHA-256 e la stessa etichetta,
#    rappresentano copie identiche della stessa immagine.
#    Ne manteniamo una sola.
# 2. Se la stessa immagine compare in split differenti,
#    utilizziamo la priorità:
#           test > validation > train
#    Esempio:
#       train/foto1.jpg  ─┐
#                         ├─ stessa immagine
#       test/foto2.jpg   ─┘
#
#       → manteniamo quella nel test
#       → proponiamo di escludere quella nel train
# Questa priorità conserva le copie già assegnate alla valutazione e toglie
# dal training quelle identiche. È una scelta di progetto, non una regola universale.
# 3. Se immagini identiche hanno ETICHETTE DIVERSE,
#    non prendiamo una decisione automatica.
#    Vengono marcate come "review_label" per revisione manuale.

def propose_duplicate_plan(image_info):
    # 1. CONTROLLI PRELIMINARI
    if not image_info["split"].isin(["train", "val", "test"]).all():
        raise ValueError("Split inatteso: la priorità è definita per train, val e test.")
    if image_info["path"].duplicated().any() or image_info["sha256"].isna().any():
        raise ValueError("Report incoerente: verificare percorsi ripetuti o hash mancanti.")

    # 2. CREAZIONE DEI GRUPPI DI DUPLICATI
    duplicate_groups = image_info.groupby("sha256").agg(
        n_files=("path", "size"),
        splits=("split", lambda values: ", ".join(sorted(set(values)))),
        n_splits=("split", "nunique"), # Numero di split differenti coinvolti
        classes=("class_name", lambda values: ", ".join(sorted(set(values)))),
        n_classes=("class_name", "nunique"), # Numero di classi differenti associate all'immagine
    )
    duplicate_groups = duplicate_groups[duplicate_groups["n_files"] > 1].reset_index()

    # 3. CREAZIONE DEL PIANO DI GESTIONE
    duplicate_plan = image_info.copy()
    duplicate_plan["priority"] = duplicate_plan["split"].map({"test": 0, "val": 1, "train": 2})
    # Ordino prima per hash e poi per priorità.
    duplicate_plan = duplicate_plan.sort_values(["sha256", "priority", "path"]).reset_index(drop=True)
    
    # 4. IDENTIFICAZIONE DELLE ETICHETTE IN CONFLITTO
    conflicting = duplicate_plan.groupby("sha256")["class_name"].transform("nunique") > 1
    
    # 5. IDENTIFICAZIONE DELLE COPIE RIPETUTE
    # keep="first" > considera la prima come non duplicata
    repeated = duplicate_plan.duplicated("sha256", keep="first")

    # 6. ASSEGNAZIONE AZIONI
    # AZIONE = inizialmente tutte le immagini vengono mantenute.
    duplicate_plan["action"] = "keep"
    # AZIONE = è una copia, proponi di escluderla
    duplicate_plan.loc[repeated, "action"] = "exclude_duplicate"
    # AZIONE = stessa immagine con classi diverse, controlla manualmente
    duplicate_plan.loc[conflicting, "action"] = "review_label"
    # elimino la colonna temporanea priority
    duplicate_plan = duplicate_plan.drop(columns="priority")

    # Controllo che non esisteano duplicati per le immagini 'keep'
    retained_images = duplicate_plan[duplicate_plan["action"] == "keep"]
    assert not retained_images["sha256"].duplicated().any()

    return duplicate_groups, duplicate_plan


In [ ]:
duplicate_groups, duplicate_plan = propose_duplicate_plan(image_info)
print("Gruppi di duplicati esatti:", len(duplicate_groups))
print("Gruppi presenti in più split:", int((duplicate_groups["n_splits"] > 1).sum()))
print("Gruppi con etichette diverse:", int((duplicate_groups["n_classes"] > 1).sum()))
print("\nFile per azione proposta e split:")
# Per ogni split (train, val, test), quante immagini hanno ciascuna azione (keep, exclude_duplicate, review_label)?
print(pd.crosstab(duplicate_plan["split"], duplicate_plan["action"]))

In [ ]:
# ============================================================
# CELLA 5.3 — Effetto della deduplicazione sulle classi
# ============================================================
# Prima di applicare realmente la proposta controllo
# come cambierebbe il numero di immagini di ogni classe.
#
# >> IMPO!!! Perché il dataset iniziale è perfettamente
# bilanciato e la rimozione dei duplicati potrebbe introdurre
# un certo sbilanciamento.

# Numero di immagini per split e classe prima e dopo
before_counts = image_info.groupby(["split", "class_name"]).size().rename("prima")
after_counts = duplicate_plan[duplicate_plan["action"] == "keep"].groupby(
    ["split", "class_name"]).size().rename("dopo_proposta")

count_comparison = pd.concat([before_counts, after_counts], axis=1).fillna(0).astype(int)
count_comparison["esclusi_proposti"] = count_comparison["prima"] - count_comparison["dopo_proposta"]
print("\nConteggi per classe e split:")
print(count_comparison.to_string())
if (count_comparison["dopo_proposta"] == 0).any():
    print("ATTENZIONE: almeno una classe resterebbe vuota. Non applicare questa proposta senza revisione.")

In [ ]:
# ============================================================
# SALVATAGGIO DEI REPORT
# ============================================================
os.makedirs("dataset_audit", exist_ok=True)
duplicate_groups.to_csv("dataset_audit/duplicate_groups.csv", index=False)
duplicate_plan.to_csv("dataset_audit/duplicate_plan.csv", index=False)
count_comparison.to_csv("dataset_audit/counts_after_proposal.csv")

# ============================================================
# REVISIONE VISIVA DELLE ETICHETTE IN CONFLITTO
# ============================================================

# Se la stessa immagine compare con etichette differenti,
# mostriamo l'immagine e tutte le etichette associate.
conflict_groups = duplicate_groups[duplicate_groups["n_classes"] > 1]
# Mostro i primi 12 gruppi. E' stato messo questo limite 
# ma al momento basta perchè sono pochi.
for row in conflict_groups.head(12).itertuples():
    members = duplicate_plan[duplicate_plan["sha256"] == row.sha256]
    print("\nGruppo:", row.sha256)
    print(members[["split", "class_name", "path"]].to_string(index=False))
    # Visualizzo l'immagine singola
    with Image.open(members.iloc[0]["path"]) as img:
        plt.figure(figsize=(5, 4))
        plt.imshow(img.convert("RGB"))
        plt.title(f"Etichette da verificare: {row.classes}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close()
# Se ci fossero più di 12 conflitti,
# il report CSV permetterebbe comunque di consultarli tutti.
if len(conflict_groups) > 12:
    print("Mostrati i primi 12 gruppi in conflitto; il CSV contiene tutti i gruppi.")
print("\nPROPOSTA SOLTANTO: i dati e i loader restano quelli originali.")
print("Prima del training definitivo, applicare una scelta documentata e ripetere audit e conteggi.")

### Analisi delle etichette in conflitto

Il controllo dei duplicati ha individuato **2 gruppi di immagini identiche associate a etichette differenti**. Questi casi sono stati visualizzati manualmente per verificare quale etichetta fosse corretta.

L'ispezione ha evidenziato che il problema non riguarda necessariamente solo la presenza di duplicati: in almeno un caso, infatti, l'immagine mostra un alimento che **non sembra appartenere a nessuna delle etichette assegnate**. In particolare, un'immagine visivamente riconducibile a una pizza compare nel dataset con le etichette `Donut` e `Sandwich`.

Questo risultato suggerisce la possibile presenza di **errori di etichettatura (label noise)** anche in immagini che non hanno duplicati. Il controllo tramite SHA-256, infatti, può rilevare un conflitto solo quando la stessa immagine compare più volte con etichette diverse, ma non può stabilire se l'etichetta di una singola immagine sia semanticamente corretta.

Per questo motivo viene effettuato anche un **controllo visivo su un campione casuale più ampio di immagini per ciascuna classe**, con l'obiettivo di verificare se gli errori osservati siano casi isolati oppure indichino un problema più diffuso nel dataset.

In [ ]:
# ============================================================
# CELLA 5.4 — Controllo visivo delle etichette
# ============================================================
# Visualizziamo un campione casuale di immagini per ogni classe.
# Lo scopo è verificare se sono presenti immagini chiaramente
# incompatibili con l'etichetta assegnata.
#
# Questo controllo non garantisce che tutte le 14.000 etichette
# siano corrette, ma permette di individuare eventuali problemi
# evidenti o sistematici nel dataset.

import random

N_SAMPLES = 20
for class_name in RAW_CLASSES:
    # Seleziono tutte le immagini appartenenti alla classe
    class_images = image_info[
        image_info["class_name"] == class_name
    ]
    # Estraggo casualmente fino a 20 immagini
    samples = class_images.sample(
        n=min(N_SAMPLES, len(class_images)),
        random_state=42
    )
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    axes = axes.flatten()
    for ax, (_, row) in zip(axes, samples.iterrows()):
        with Image.open(row["path"]) as img:
            ax.imshow(img.convert("RGB"))
        ax.axis("off")
    plt.suptitle(
        f"Controllo etichette — {DISPLAY_NAME[class_name]}",
        fontsize=14
    )
    plt.tight_layout()
    plt.show()
    plt.close()

### Controllo qualitativo delle etichette

A seguito dei conflitti individuati durante l'analisi dei duplicati, è stato effettuato un ulteriore controllo visivo su un campione casuale di **20 immagini per ciascuna delle 14 classi**, per un totale di **280 immagini**.

Nel campione osservato, le immagini risultano nel complesso coerenti con le rispettive classi e **non emerge un problema sistematico evidente di errata etichettatura**.

Questo controllo rimane comunque campionario e non permette di garantire la correttezza di tutte le 14.000 etichette. Tuttavia, insieme all'analisi dei duplicati, suggerisce che le anomalie individuate possano essere circoscritte ai pochi casi segnalati, che vengono quindi gestiti tramite revisione manuale.

### Decisione sui conflitti e applicazione del piano

A seguito dell'analisi dei duplicati sono stati individuati **due gruppi di immagini identiche associate a etichette differenti**, per un totale di **5 file**.

La revisione visiva ha mostrato che:
- un gruppo rappresentava una **pizza**, alimento non presente tra le 14 classi del dataset, ma associato a etichette incompatibili;
- l'altro mostrava un **hamburger**, immagine ambigua rispetto alle classi a cui era stato assegnato.

La decisione è stata quindi quella di **escludere tutti e 5 i file**, evitando di assegnare arbitrariamente queste immagini a una delle classi disponibili.

Per gli altri duplicati esatti, associati invece alla stessa classe, viene conservata **una sola copia** ed escluse le successive, in modo da evitare che la stessa immagine sia presente più volte nel dataset.

La pulizia viene applicata attraverso una lista dei file ammessi (`accepted_paths`), che sarà utilizzata dai successivi loader di training, validation e test. **I file originali non vengono cancellati o modificati sul disco.**

Al termine della procedura viene inoltre verificato che:
- non rimangano duplicati esatti tra le immagini conservate;
- non rimangano immagini con etichette in conflitto;
- tutte le classi continuino a essere rappresentate nei rispettivi split.

La rimozione dei duplicati può modificare leggermente il bilanciamento iniziale delle classi; per questo motivo la distribuzione viene nuovamente controllata dopo la pulizia.



In [ ]:
# ============================================================
# CELLA 5.5 — Accettazione e applicazione del piano
# ============================================================
# Questa cella rende effettiva, per i successivi DataLoader,
# la proposta di pulizia costruita nelle celle precedenti.
# - "keep"              -> immagine mantenuta
# - "exclude_duplicate" -> copia duplicata esclusa
# - "review_label"      -> immagine con etichetta ambigua/errata esclusa
# ATTENZIONE:
# i file originali NON vengono cancellati o spostati.
# Viene creata semplicemente una lista contenente i percorsi
# delle immagini che potranno essere utilizzate successivamente.

def accept_duplicate_plan(image_info, duplicate_plan):
    # 1. VERIFICA CHE IL PIANO CORRISPONDA AL DATASET ANALIZZATO
    columns = ["path", "split", "class_name", "sha256"]
    # Ogni percorso deve comparire una sola volta nel piano.
    if duplicate_plan["path"].duplicated().any():
        raise ValueError("Il piano contiene percorsi ripetuti.")
    # Info originali
    original = image_info[columns].sort_values("path").reset_index(drop=True)
    # Info del piano proposto
    planned = duplicate_plan[columns].sort_values("path").reset_index(drop=True)
    
    # I due DataFrame devono contenere esattamente le stesse immagini.
    if not original.equals(planned):
        raise ValueError("Il piano non corrisponde all'audit corrente: rieseguire la cella 5.2.")
    
    # 2. VERIFICA DELLE AZIONI
    if not duplicate_plan["action"].isin(["keep", "exclude_duplicate", "review_label"]).all():
        raise ValueError("Azione non riconosciuta nel piano.")
    
    # 3. CREAZIONE DEL DATASET PULITO
    # Conservo SOLO le immagini contrassegnate come "keep".
    retained_images = duplicate_plan[duplicate_plan["action"] == "keep"].copy()

    # 4. CONTROLLO DEI CONFLITTI DI ETICHETTA nel nuovo dataset
    conflicting = image_info.groupby("sha256")["class_name"].nunique()
    if retained_images["sha256"].isin(conflicting[conflicting > 1].index).any():
        raise ValueError("Il piano conserva immagini con etichette in conflitto.")
    
    # 5. CONTROLLO CHE NON SIANO RIMASTI DUPLICATI nel nuovo dataset
    if retained_images["sha256"].duplicated().any():
        raise ValueError("Il piano conserva ancora duplicati esatti.")
    expected = image_info.groupby(["split", "class_name"]).size()
    
    # 6. CONTROLLO CHE NESSUNA CLASSE DIVENTI VUOTA
    actual = retained_images.groupby(["split", "class_name"]).size().reindex(expected.index, fill_value=0)
    if (actual == 0).any():
        raise ValueError("La pulizia lascerebbe una classe vuota: rivedere il piano.")
    
    # 7. CONTROLLO CHE I FILE ESISTANO ANCORA SUL DISCO
    missing = [path for path in retained_images["path"] if not os.path.isfile(path)]
    if missing:
        raise ValueError(f"File non più presenti: {missing[:3]}. Ripetere l'audit.")
    
    # 8. CREAZIONE DELLA LISTA DEI FILE ACCETTATI
    # Usati per stabilire le immagini che possono appartenere al DataLoader
    accepted_paths = tuple(os.path.normcase(os.path.abspath(path)) for path in retained_images["path"])
    return retained_images, accepted_paths


In [ ]:
# ============================================================
# APPLICAZIONE DEL PIANO
# ============================================================
accepted_paths = ()
retained_images, accepted_paths = accept_duplicate_plan(image_info, duplicate_plan)

# ============================================================
# SALVATAGGIO DELLA DECISIONE FINALE
# ============================================================
os.makedirs("dataset_audit", exist_ok=True)
accepted_plan = duplicate_plan.copy()
accepted_plan["decision"] = accepted_plan["action"].map({
    "keep": "conservato",
    "exclude_duplicate": "escluso: copia identica, priorità test > val > train",
    "review_label": "escluso: etichetta incompatibile o ambigua dopo revisione visiva",
})

accepted_plan.to_csv("dataset_audit/accepted_duplicate_plan.csv", index=False)
retained_images.to_csv("dataset_audit/retained_images.csv", index=False)
clean_counts = pd.crosstab(retained_images["class_name"], retained_images["split"]).reindex(
    index=RAW_CLASSES, columns=["train", "val", "test"], fill_value=0)
clean_counts.to_csv("dataset_audit/clean_counts.csv")

print("Piano applicato ai dati dei successivi loader.")
print(f"Immagini originali: {len(image_info)} | Conservate: {len(retained_images)} | "
      f"Escluse: {len(image_info) - len(retained_images)}")
print(clean_counts.to_string())
print("Duplicati esatti nei record conservati: 0. Nessun file originale modificato.")

In [ ]:
# Nuovo controllo visivo della distribuzione dopo l'applicazione del piano
plot_train_distribution(clean_counts, "Distribuzione delle 14 classi nel training set dopo la pulizia")

### Distribuzione delle classi dopo la pulizia

Dopo l'applicazione definitiva del piano di pulizia vengono conservate **13.810 immagini su 14.000**, con l'esclusione di **190 immagini** tra copie duplicate e immagini appartenenti ai gruppi con etichette incompatibili o ambigue. Nessun file originale viene cancellato o modificato e, nei dati conservati, **non rimangono duplicati esatti**.

Le esclusioni non sono distribuite uniformemente tra le classi. Le **6 classi** `Apple Pie`, `Cheesecake`, `Chicken Curry`, `Ice Cream`, `Omelette` e `Sushi` mantengono infatti la distribuzione originale di **640 immagini nel train, 160 nella validation e 200 nel test**. Le altre classi subiscono invece una riduzione dovuta alla presenza di duplicati.

Le riduzioni più evidenti nel training set riguardano:

| Classe | Train dopo pulizia | Riduzione |
|---|---:|---:|
| Fries | 585 | −55 |
| Taquito | 608 | −32 |
| Baked Potato | 614 | −26 |
| Hot Dog | 627 | −13 |
| Sandwich | 628 | −12 |
| Crispy Chicken | 629 | −11 |

Le altre riduzioni sono più contenute: `Donut` passa a 635 immagini e `Taco` a 638. Anche validation e test subiscono solo piccole variazioni, con un minimo rispettivamente di **151** e **197 immagini per classe**.

Il dataset, inizialmente perfettamente bilanciato, presenta quindi dopo la pulizia un **lieve sbilanciamento**. Nel training set la classe meno numerosa è `Fries` con 585 immagini, rispetto alle 640 delle classi più numerose: conserva quindi circa il **91%** degli esempi disponibili nelle classi maggiori.

Lo sbilanciamento rimane contenuto e, in questa fase, non rende necessario introdurre tecniche specifiche di riequilibrio. Sarà comunque opportuno osservare anche le **metriche per singola classe**, per verificare che le classi con meno esempi non presentino prestazioni sensibilmente inferiori.

## 4. Configurazione centralizzata

Tutti i parametri modificabili vengono raccolti in un'unica classe `Config`. Il vantaggio principale è che il confronto tra esperimenti diventa esplicito.

In [ ]:
# ============================================================
# CELLA 7 — Classe Config: configurazione centralizzata
# ============================================================

@dataclass
class Config:
    """
    Classe di configurazione centrale del progetto.
    L'idea è raccogliere qui tutti i parametri modificabili dei vari test.
    """
    # --- Identificazione dell'esperimento ---
    experiment_name: str = "baseline"

    # --- Percorsi del dataset ---
    # Struttura attesa: dataset_root/train/, dataset_root/val/, dataset_root/test/
    # con una sottocartella per classe in ciascuno split (già così nel dataset fornito).
    dataset_root: str = "datasets/dataset_food_classification"
    # Lista esplicita prodotta dalla cella 5.5; vuota = training bloccato.
    allowed_paths: tuple = ()
    train_dir: str = "train"
    val_dir: str = "val"
    test_dir: str = "test"

    # --- Modello pretrained ---
    # efficientnet_b0: buon compromesso accuratezza/velocità per dataset di dimensioni
    # contenute su GPU singola (Colab). Alternative valutabili: resnet18, vit_small_patch16_224.
    model_name: str = "efficientnet_b0"
    num_classes: int = len(RAW_CLASSES)
    # Lato dell'immagine in input (pixel). None = risoluzione nativa del modello
    # pretrained (224 per EfficientNet-B0). Valori più alti conservano più dettagli.
    img_size: int = None

    # --- Iperparametri di training ---
    batch_size: int = 32
    epochs: int = 25               # budget massimo per esperimento (early stopping può fermare prima)
    lr: float = 5e-4               # identico nei confronti controllati
    weight_decay: float = 1e-4
    num_workers: int = 2
    use_amp: bool = True           # Mixed precision su GPU
    seed: int = 42

    # --- Strategia di fine-tuning ---
    freeze_backbone: bool = True
    progressive_unfreeze: bool = False
    unfreeze_epoch: int = 4

    # --- Scheduler del learning rate ---
    use_cosine_scheduler: bool = False   # False -> ReduceLROnPlateau, True -> CosineAnnealingLR

    # --- Regolarizzazione avanzata ---
    label_smoothing: float = 0.0   # >0: target "soft" invece di one-hot (previene overconfidence)
    mixup_alpha: float = 0.0       # >0: attiva Mixup (immagini ed etichette "ibride")

    # --- Livello di data augmentation ---
    # "basic"    -> crop casuale conservativo + flip orizzontale
    # "moderate" -> + color jitter + RandAugment leggero + random erasing
    # "strong"   -> + color jitter forte + RandAugment intenso + random erasing
    augmentation: str = "basic"

    # --- Early stopping ---
    patience: int = 4

    # --- Salvataggio ---
    best_model_path: str = "best_model.pth"
    # True: se esiste già un checkpoint completo di questo esperimento, con la stessa
    # configurazione, il training viene saltato e il modello ricaricato dal file.
    # Per forzare un nuovo training impostare False o cancellare il checkpoint.
    resume_from_checkpoint: bool = True


## 5. Classe `ExperimentRunner`: gestione dell'intero ciclo di un esperimento

Questa classe raccoglie tutte le operazioni necessarie per eseguire un esperimento: caricamento dati, costruzione del modello, training, validazione, early stopping. Separare la logica in metodi distinti (`build_dataloaders`, `build_model`, `build_optimizer`...) rende più facile isolare e modificare singole componenti senza toccare il resto della pipeline.

In [ ]:
# ============================================================
# CELLA 8 — FilteredImageFolder: ImageFolder limitato ai file accettati
# ============================================================
# Carica soltanto le immagini conservate dal piano della cella 5.5.

class FilteredImageFolder(ImageFolder):
    def __init__(self, root, transform, allowed_paths=()):
        super().__init__(root, transform=transform)
        if not allowed_paths:
            raise ValueError("Eseguire la cella 5.5 e ricreare le configurazioni prima del training.")
        allowed = set(allowed_paths)
        root_path = os.path.normcase(os.path.abspath(root)) + os.sep
        expected = {path for path in allowed if path.startswith(root_path)}
        self.samples = [(path, label) for path, label in self.samples
                        if os.path.normcase(os.path.abspath(path)) in allowed]
        found = {os.path.normcase(os.path.abspath(path)) for path, _ in self.samples}
        if found != expected or not found:
            raise ValueError("I file del loader non corrispondono al piano: ripetere audit e accettazione.")
        if set(label for _, label in self.samples) != set(range(len(self.classes))):
            raise ValueError("Almeno una classe non ha immagini dopo il filtro.")
        self.targets = [label for _, label in self.samples]
        self.imgs = self.samples


In [ ]:
# ============================================================
# CELLA 9 — ExperimentRunner
# ============================================================
# Questa classe centralizza tutte le operazioni necessarie per eseguire un esperimento:
# caricamento dati, costruzione del modello, training, validazione ed early stopping.
# Separare la logica in metodi distinti rende più facile isolare e modificare singole
# componenti senza toccare il resto della pipeline.
# ============================================================

class ExperimentRunner:
    def __init__(self, config: Config, device=None):
        self.config = config
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # GradScaler: necessario per usare Automatic Mixed Precision (AMP) in modo stabile.
        # AMP usa float16 dove possibile (può ridurre memoria e tempi) ma può causare
        # underflow numerici nei gradienti piccoli: lo scaler li compensa automaticamente.
        self.scaler = torch.amp.GradScaler(
            "cuda", enabled=(self.config.use_amp and self.device.type == "cuda")
        )

        # Loader e metadati del dataset: inizializzati in build_dataloaders()
        self.train_loader = None
        self.val_loader = None
        self.test_loader = None
        self.class_names = None

        # Componenti del modello: inizializzate nei rispettivi metodi build_*()
        self.model = None
        self.train_criterion = None # loss usata durante il training (può includere smoothing/mixup)
        self.eval_criterion = nn.CrossEntropyLoss()   # loss usata in validation/test sempre standard
        self.optimizer = None
        self.scheduler = None
        self.mixup_fn = None
        self.data_cfg = None

    # ----------------------------------------------------------
    # Utility
    # ----------------------------------------------------------

    def set_seed(self):
        set_seed(self.config.seed)

    def load_data_config(self):
        """
        Recupera la configurazione nativa del modello pretrained da timm:
        - mean e std usati durante il pretraining su ImageNet
        - input_size attesa dal backbone (es. 224x224 per EfficientNet-B0)
        - crop_pct: percentuale di crop per la valutazione
        - interpolation: metodo di resize (bilinear, bicubic...)

        È fondamentale usare le stesse normalizzazioni del pretraining:
        i feature extractor pretrained sono ottimizzati per queste statistiche.
        Usare statistiche diverse riduce le performance del transfer learning.
        """
        model_for_cfg = timm.create_model(self.config.model_name, pretrained=True)
        # Se img_size è impostato, sostituisce la risoluzione nativa del modello.
        # Mean e std restano quelle del pretraining.
        overrides = {}
        if self.config.img_size is not None:
            overrides["input_size"] = (3, self.config.img_size, self.config.img_size)
        self.data_cfg = resolve_data_config(overrides, model=model_for_cfg)
        return self.data_cfg

    def get_transforms(self):
        """
        Crea le trasformazioni usando timm.create_transform.
        Il livello di augmentation è controllato da config.augmentation.

        Tutti i livelli partono da crop casuale conservativo (scale 0.7-1.0, ratio 0.85-1.15)
        e flip orizzontale (50%). Il flip verticale è sempre disattivato (vflip=0.0).
        - basic:    solo crop casuale + flip orizzontale, adatto alla baseline
        - moderate: + color jitter 0.2 + RandAugment leggero (m7) + random erasing (10%)
        - strong:   + color jitter 0.3 + RandAugment intenso (m9) + random erasing (20%)

        IMPORTANTE: le trasformazioni di augmentation vengono applicate SOLO al training set.
        Per validation e test si usa sempre eval_tfms (resize, crop centrale e normalizzazione,
        senza casualità), perché questi split devono rappresentare le condizioni reali d'uso.
        """
        if self.data_cfg is None:
            self.load_data_config()

        # Parametri condivisi tra training e valutazione:
        # normalizzazione e dimensioni devono essere identiche al pretraining
        common_kwargs = {
            "input_size": self.data_cfg["input_size"],
            "mean": self.data_cfg["mean"],
            "std": self.data_cfg["std"],
            "interpolation": self.data_cfg["interpolation"],
            "crop_pct": self.data_cfg["crop_pct"],
        }

        if self.config.augmentation == "basic":
            # Crop casuale conservativo + flip orizzontale: simula inquadrature diverse del piatto.
            # Nessuna variazione di colore, nessuna policy automatica, nessun erasing.
            train_tfms = create_transform(
                is_training=True, 
                scale=(0.7, 1.0), 
                ratio=(0.85, 1.15), 
                hflip=0.5, 
                vflip=0.0, 
                color_jitter=0.0,
                auto_augment=None, 
                re_prob=0.0, 
                **common_kwargs
            )
        elif self.config.augmentation == "moderate":
            # color_jitter=0.2: variazioni leggere di luminosità, contrasto e saturazione.
            # RandAugment (rand-m7): policy automatica di augmentation con magnitudo 7.
            # re_prob=0.10 → random erasing: copre una regione casuale dell'immagine (10% delle volte).
            # Simula occlusioni parziali del piatto (posate, mani, altri oggetti).
            train_tfms = create_transform(
                is_training=True, 
                scale=(0.7, 1.0), 
                ratio=(0.85, 1.15), 
                hflip=0.5, 
                vflip=0.0, 
                color_jitter=0.2,
                auto_augment="rand-m7-mstd0.5-inc1", 
                re_prob=0.10,
                re_mode="pixel", 
                re_count=1, 
                **common_kwargs
            )
        elif self.config.augmentation == "strong":
            # Augmentation più aggressiva: color jitter 0.3, RandAugment magnitudo 9, random erasing 20%.
            # Il flip verticale resta disattivato come negli altri livelli.
            # Aumenta la variabilità del training set; va verificato che non distorca
            # colore e aspetto dei piatti al punto da renderli irriconoscibili.
            train_tfms = create_transform(
                is_training=True, 
                scale=(0.7, 1.0), 
                ratio=(0.85, 1.15), 
                hflip=0.5, 
                vflip=0.0, 
                color_jitter=0.3,
                auto_augment="rand-m9-mstd0.5-inc1", 
                re_prob=0.20,
                re_mode="pixel", 
                re_count=1, 
                **common_kwargs
            )
        else:
            raise ValueError("augmentation deve essere 'basic', 'moderate' o 'strong'")

        # Trasformazioni di valutazione: resize, crop centrale e normalizzazione, senza casualità
        eval_tfms = create_transform(is_training=False, **common_kwargs)

        return train_tfms, eval_tfms

    def build_dataloaders(self):
        """
        Costruisce dataset e dataloader per train, validation e test.

        FilteredImageFolder carica soltanto i file accettati dalla cella 5.5 e
        rileva le classi dalle sotto-cartelle:
            dataset_root/train/Baked Potato/ → classe 0
            dataset_root/train/Crispy Chicken/ → classe 1
            ...
        L'ordine alfabetico determina l'indice numerico della classe.
        Il metodo verifica che classi e indici coincidano nei tre split e che
        siano 14, come indicato in config.num_classes.
        """
        train_path = os.path.join(self.config.dataset_root, self.config.train_dir)
        val_path = os.path.join(self.config.dataset_root, self.config.val_dir)
        test_path = os.path.join(self.config.dataset_root, self.config.test_dir)

        train_tfms, eval_tfms = self.get_transforms()

        train_dataset = FilteredImageFolder(train_path, transform=train_tfms, allowed_paths=self.config.allowed_paths)
        val_dataset = FilteredImageFolder(val_path, transform=eval_tfms, allowed_paths=self.config.allowed_paths)
        test_dataset = FilteredImageFolder(test_path, transform=eval_tfms, allowed_paths=self.config.allowed_paths)

        # Verifica di consistenza: le classi devono coincidere tra i tre split
        if train_dataset.classes != val_dataset.classes or train_dataset.classes != test_dataset.classes:
            raise ValueError("Le classi di train, validation e test non coincidono.")

        self.class_names = train_dataset.classes
        if len(self.class_names) != self.config.num_classes:
            raise ValueError("num_classes non coincide con le classi del dataset")
        if self.config.mixup_alpha > 0 and (self.config.batch_size % 2 or len(train_dataset) < self.config.batch_size):
            raise ValueError("Mixup richiede batch_size pari e almeno un batch completo")

        # shuffle=True solo per il training: mescola i campioni ad ogni epoca per evitare che il modello impari l'ordine dei dati.
        # pin_memory=True: prepara i tensori in memoria pinned per un trasferimento più veloce alla GPU (utile solo se si usa CUDA).
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            pin_memory=torch.cuda.is_available(),
            # drop_last è attivo solo con Mixup: scarta l'ultimo batch incompleto, che
            # potrebbe avere dimensione dispari e non permettere di accoppiare le immagini.
            drop_last=(self.config.mixup_alpha > 0),
        )
        self.val_loader = DataLoader(
            val_dataset, 
            batch_size=self.config.batch_size, 
            shuffle=False,
            num_workers=self.config.num_workers, 
            pin_memory=torch.cuda.is_available(),
        )
        self.test_loader = DataLoader(
            test_dataset, 
            batch_size=self.config.batch_size, 
            shuffle=False,
            num_workers=self.config.num_workers, 
            pin_memory=torch.cuda.is_available(),
        )
        return train_dataset, val_dataset, test_dataset

    # ----------------------------------------------------------
    # Modello
    # ----------------------------------------------------------
    def build_model(self):
        """
        Costruisce il modello indicato in config.model_name (EfficientNet-B0 o B2)
        utilizzando il transfer learning.

        Il modello parte dai pesi già appresi su ImageNet e la testa
        di classificazione finale viene adattata alle 14 classi del dataset.

        Se freeze_backbone=True:
        - il backbone viene congelato;
        - viene allenata soltanto la nuova testa di classificazione.
        """
        self.model = timm.create_model(
            self.config.model_name, 
            pretrained=True, 
            num_classes=self.config.num_classes
        )

        # Se nella configurazione abbiamo scelto di congelare il backbone
        if self.config.freeze_backbone:
            # Congeliamo inizialmente TUTTI i parametri del modello.
            for param in self.model.parameters():
                param.requires_grad = False
            # get_classifier() restituisce la testa finale
            # di classificazione di EfficientNet.
            # Viene riattivata per allenare la parte finale di classificazione
            for param in self.model.get_classifier().parameters():
                param.requires_grad = True
        # Salvataggio dello stato corrente del backbone.
        # True  -> backbone congelato
        # False -> backbone allenabile
        self.backbone_frozen = self.config.freeze_backbone

        self.model = self.model.to(self.device)
        return self.model

    def unfreeze_backbone(self):
        """
        Sblocca l'intero modello per permettere il fine-tuning
        anche del backbone.
        """
        for param in self.model.parameters():
            param.requires_grad = True
        # Aggiornamento della variabile che tiene traccia dello stato:
        # il backbone ora NON è più congelato.
        self.backbone_frozen = False

    # ----------------------------------------------------------
    # Loss, ottimizzatore, scheduler
    # ----------------------------------------------------------
    def build_mixup(self):
        """
        Mixup crea campioni sintetici mescolando coppie di immagini e le relative
        etichette: 
        img_mix = lambda*img_a + (1-lambda)*img_b, con lambda ~ Beta(alpha, alpha).        
        
        Anche le etichette vengono combinate con le stesse proporzioni.
        Ad esempio, mescolando al 70% un'immagine Taco e al 30% una
        Taquito, il target non sarà più una singola classe, ma una
        soft label: 0.7 Taco e 0.3 Taquito.

        In questo modo il modello impara caratteristiche più generali delle classi,
        invece di memorizzare rigidamente le immagini di training, 
        e può quindi aiutare a ridurre l'overfitting.
        """
        if self.config.mixup_alpha > 0:
            self.mixup_fn = Mixup(
                mixup_alpha=self.config.mixup_alpha, cutmix_alpha=0.0, prob=1.0,
                switch_prob=0.0, mode="batch", label_smoothing=self.config.label_smoothing,
                num_classes=self.config.num_classes,
            )
        else:
            self.mixup_fn = None

    def build_criterion(self):
        """
        Mixup attivo         -> SoftTargetCrossEntropy (etichette morbide)
        Solo label smoothing -> LabelSmoothingCrossEntropy
        Nessuna delle due     -> CrossEntropyLoss standard
        """
        if self.config.mixup_alpha > 0:
            self.train_criterion = SoftTargetCrossEntropy()
        elif self.config.label_smoothing > 0:
            self.train_criterion = LabelSmoothingCrossEntropy(smoothing=self.config.label_smoothing)
        else:
            self.train_criterion = nn.CrossEntropyLoss()

    def build_optimizer(self):
        """
        AdamW sui soli parametri allenabili (requires_grad=True).
        """
        trainable_params = [p for p in self.model.parameters() if p.requires_grad]
        self.optimizer = optim.AdamW(
            trainable_params, lr=self.config.lr, weight_decay=self.config.weight_decay
        )

    def build_scheduler(self):
        """
        Configura lo scheduler del learning rate.

        ReduceLROnPlateau (default):
        - Monitora il F1-score macro di validation (mode='max')
        - Se non migliora per 2 epoche → moltiplica lr × 0.5
        - Vantaggio: adattivo, riduce lr solo quando serve
        - Uso: adatto per training con early stopping

        CosineAnnealingLR (alternativa):
        - Riduce lr seguendo una curva a coseno, dal valore iniziale fino a 0 in T_max=epochs epoche
        - Indipendente dalle metriche di validazione
        - Uso: adatto quando si conosce in anticipo il numero di epoche
        """
        if self.config.use_cosine_scheduler:
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, 
                T_max=self.config.epochs
                )
        else:
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer, 
                mode="max", 
                factor=0.5, 
                patience=2
            )

    # ----------------------------------------------------------
    # Metriche
    # ----------------------------------------------------------
    def compute_metrics(self, y_true, y_pred, y_proba=None):
        """
        Calcola accuracy, F1 macro e, se disponibili le probabilità, top-3 accuracy.

        - F1 macro: media del F1 di ogni classe, senza ponderazione per frequenza.
          È la metrica principale di selezione perché pesa allo stesso modo tutte
          le 14 classi, anche quelle con meno immagini dopo la pulizia.
        - Top-3 accuracy (solo se y_proba è fornito): quota di immagini la cui classe
          corretta è tra le 3 più probabili. Con 14 classi misura "quanto ci arriva
          vicino" il modello, complementare all'accuracy (che considera solo la
          predizione col punteggio più alto).
        """
        acc = accuracy_score(y_true, y_pred)
        f1_macro = f1_score(y_true, y_pred, average="macro")
        top3 = None
        if y_proba is not None:
            top3 = top_k_accuracy_score(
                y_true, y_proba, k=3, labels=list(range(self.config.num_classes))
            )
        return acc, f1_macro, top3

    # ----------------------------------------------------------
    # Training / validazione
    # ----------------------------------------------------------
    def train_one_epoch(self):
        """
        Esegue un'epoca di training e restituisce loss, accuracy e F1 macro di training.

        Passi per ogni batch:
        1. Trasferimento immagini e label sul device (GPU)
        2. Eventuale applicazione di Mixup (crea campioni sintetici)
        3. Reset dei gradienti del batch precedente
        4. Forward pass con AMP (autocast riduce la precisione dove sicuro)
        5. Backpropagation scalata e aggiornamento dei pesi (lo scaler evita underflow con AMP)

        model.train() attiva Dropout (azzera neuroni casuali) e BatchNorm in modalità
        training (usa e aggiorna le statistiche del batch corrente). Con il backbone
        congelato solo la testa resta in modalità training: il feature extractor torna
        in eval, così anche le sue statistiche BatchNorm non cambiano.

        Con Mixup accuracy e F1 di training valgono NaN: le etichette sono miscelate.
        """
        self.model.train()
        if self.backbone_frozen:
            # Blocca anche statistiche BatchNorm e dropout del feature extractor.
            self.model.eval()
            self.model.get_classifier().train()
        running_loss = 0.0
        y_true, y_pred = [], []
        samples_seen = 0

        for images, labels in self.train_loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            # Salva le etichette originali prima di applicare Mixup
            # (Mixup produce etichette morbide usate solo per la loss, 
            # non per calcolare le metriche di classificazione)
            original_labels = labels.clone()
            if self.mixup_fn is not None:
                images, labels = self.mixup_fn(images, labels)

            # zero_grad() prima di ogni batch: PyTorch accumula i gradienti per default;
            # non azzerarli causerebbe la somma con i gradienti del batch precedente (errore comune per i principianti)
            self.optimizer.zero_grad()

            # autocast: usa float16 per le operazioni supportate (Conv2d, Linear...) riducendo memoria e accelerando il calcolo sulla GPU
            with torch.autocast(device_type=self.device.type,
                                 enabled=(self.config.use_amp and self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.train_criterion(outputs, labels)

            # scaler.scale(loss).backward(): calcola i gradienti scalando la loss per evitare underflow numerici con float16
            self.scaler.scale(loss).backward()
            # scaler.step(): de-scala i gradienti e aggiorna i pesi
            self.scaler.step(self.optimizer)
            # scaler.update(): adatta il fattore di scala per il prossimo step
            self.scaler.update()

            running_loss += loss.item() * images.size(0)
            # argmax: prende l'indice della classe con il logit più alto
            preds = torch.argmax(outputs, dim=1)

            samples_seen += images.size(0)
            # Se Mixup NON è attivo, le immagini mantengono la loro etichetta
            # originale e quindi possiamo confrontare direttamente classe reale
            # e classe predetta.
            # - vengono salvate le etichette REALI del batch.
            # - vengono salvate le etichette PREDETTE del batch.
            #
            # Se Mixup fosse attivo, invece, i target sarebbero soft label
            # (es. 70% Taco e 30% Taquito) e non una singola classe reale.
            # Per questo non calcoliamo la normale accuracy/F1 sui batch con Mixup.
            if self.mixup_fn is None:
                y_true.extend(original_labels.detach().cpu().numpy())
                y_pred.extend(preds.detach().cpu().numpy())

        if samples_seen == 0:
            raise ValueError("Nessun batch di training disponibile")
        epoch_loss = running_loss / samples_seen
        epoch_acc, epoch_f1 = float("nan"), float("nan")

        # Se Mixup NON è attivo possiamo confrontare direttamente
        # le classi reali con quelle predette.
        #
        # Con Mixup questo confronto non sarebbe corretto perché il target
        # può rappresentare contemporaneamente più classi con pesi diversi.
        if self.mixup_fn is None:
            epoch_acc, epoch_f1, _ = self.compute_metrics(y_true, y_pred)
        return epoch_loss, epoch_acc, epoch_f1

    @torch.no_grad()
    def evaluate_loader(self, loader):
        """
        Esegue una passata di valutazione su un DataLoader (validation durante il training).

        @torch.no_grad(): disabilita il calcolo del grafo dei gradienti, riducendo
        memoria e tempi: la backpropagation non serve in fase di valutazione.

        model.eval(): disattiva il Dropout (tutti i neuroni attivi) e fa usare a
        BatchNorm le statistiche globali accumulate nel training, per ottenere
        previsioni deterministiche e stabili.

        La loss è sempre la CrossEntropy standard, anche se il training usa label
        smoothing o Mixup. Restituisce loss, accuracy, F1 macro, top-3 e predizioni.
        """
        self.model.eval()
        running_loss = 0.0
        y_true, y_pred, y_proba = [], [], []

        for images, labels in loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            with torch.autocast(device_type=self.device.type,
                                 enabled=(self.config.use_amp and self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.eval_criterion(outputs, labels)   # sempre CrossEntropy standard

            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs.float(), dim=1)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.detach().cpu().numpy())
            y_pred.extend(preds.detach().cpu().numpy())
            y_proba.extend(probs.detach().cpu().numpy())

        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc, epoch_f1, epoch_top3 = self.compute_metrics(y_true, y_pred, y_proba)

        return {
            "loss": epoch_loss, 
            "acc": epoch_acc, 
            "f1_macro": epoch_f1, 
            "top3": epoch_top3,
            "y_true": y_true, 
            "y_pred": y_pred, 
            "y_proba": y_proba,
        }

    def run(self):
        """
        Esegue l'intero ciclo dell'esperimento:
        1. Preparazione (seed, data config, dataloader, modello, loss, optimizer, scheduler).
           Se esiste già un checkpoint completo con la stessa configurazione, il modello
           viene ricaricato da lì e il training saltato (ripresa dopo una disconnessione).
        2. Loop per ogni epoca: eventuale sblocco del backbone → train_one_epoch →
           evaluate_loader(val) → scheduler step
        3. Early stopping: conserva i pesi con il miglior F1 di validation e termina
           dopo `patience` epoche senza miglioramento (mai prima dello sblocco)
        4. Ripristino del miglior modello e salvataggio del checkpoint su disco
           (pesi, storico e configurazione, nella cartella della cella 1B)

        METRICA DI SELEZIONE: F1-score macro su validation set.
        Il test set NON viene mai toccato durante questo loop.

        Restituisce un dizionario con il modello migliore, lo storico
        delle metriche per ogni epoca, i loader e le statistiche finali.
        """

        if self.config.epochs < 1:
            raise ValueError("epochs deve essere positivo")
        self.set_seed()
        self.load_data_config()
        train_dataset, val_dataset, test_dataset = self.build_dataloaders()
        self.build_model()

        # Ripresa: se l'esperimento è già stato completato in una sessione precedente,
        # ricarichiamo pesi e storico dal checkpoint invece di riaddestrare.
        checkpoint = self.load_checkpoint()
        if checkpoint is not None:
            self.model.load_state_dict(checkpoint["model_state"])
            print(f"Esperimento: {self.config.experiment_name}")
            print(f"[INFO] Checkpoint trovato in {self.config.best_model_path}: training saltato.")
            print(f"Miglior epoca: {checkpoint['best_epoch']} | "
                  f"Miglior Val F1 Macro: {checkpoint['best_val_f1']:.4f}")
            return self.build_result(checkpoint["history"], checkpoint["best_epoch"], checkpoint["best_val_f1"])

        self.build_mixup()
        self.build_criterion()
        self.build_optimizer()
        self.build_scheduler()

        history = {"train_loss": [], "train_acc": [], "train_f1": [],
                   "val_loss": [], "val_acc": [], "val_f1": [], "val_top3": []}

        best_state = None
        best_val_f1 = -1.0
        best_epoch = 0
        early_stop_counter = 0

        print(f"Esperimento: {self.config.experiment_name}")
        print(f"Classi ({len(self.class_names)}): {self.class_names}")
        print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")

        for epoch in range(1, self.config.epochs + 1):
            start_time = time.time()

            # Fine-tuning progressivo: sblocca il backbone all'epoca specificata
            if self.config.progressive_unfreeze and epoch == self.config.unfreeze_epoch:
                print(f"\n[INFO] Sblocco completo del backbone all'epoca {epoch}.")
                self.unfreeze_backbone()
                # Ricostruzione dell'optimizer e dello scheduler per includere i nuovi parametri
                self.build_optimizer()
                self.build_scheduler()
                early_stop_counter = 0

            train_loss, train_acc, train_f1 = self.train_one_epoch()
            val_metrics = self.evaluate_loader(self.val_loader)

            # ReduceLROnPlateau vuole il valore della metrica (F1);
            # CosineAnnealingLR non richiede argomenti
            if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_metrics["f1_macro"])
            else:
                self.scheduler.step()

            # Aggiornamento dello storico per i grafici
            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["train_f1"].append(train_f1)
            history["val_loss"].append(val_metrics["loss"])
            history["val_acc"].append(val_metrics["acc"])
            history["val_f1"].append(val_metrics["f1_macro"])
            history["val_top3"].append(val_metrics["top3"])

            elapsed = time.time() - start_time
            print(f"\nEpoca [{epoch}/{self.config.epochs}] - {elapsed:.1f}s")
            print(f"Train -> Loss: {train_loss:.4f} | Accuracy: {train_acc:.4f} | F1 Macro: {train_f1:.4f}")
            print(f"Val   -> Loss: {val_metrics['loss']:.4f} | Accuracy: {val_metrics['acc']:.4f} | "
                  f"F1 Macro: {val_metrics['f1_macro']:.4f} | Top-3 Acc: {val_metrics['top3']:.4f}")

            # Early stopping: se il F1 di validation migliora conserva i pesi in memoria
            if val_metrics["f1_macro"] > best_val_f1:
                best_val_f1 = val_metrics["f1_macro"]
                best_epoch = epoch
                # Salva una copia profonda dello state_dict (solo i pesi numerici)
                best_state = copy.deepcopy(self.model.state_dict())
                early_stop_counter = 0
            else:
                early_stop_counter += 1

            may_stop = not self.config.progressive_unfreeze or epoch >= self.config.unfreeze_epoch
            if may_stop and early_stop_counter >= self.config.patience:
                print(f"\n[INFO] Early stopping attivato. Nessun miglioramento da {self.config.patience} epoche.")
                break

        # Ripristina i pesi del miglior modello trovato durante il training
        # e salva il checkpoint completo (pesi, storico e configurazione).
        if best_state is not None:
            self.model.load_state_dict(best_state)
            self.save_checkpoint(best_state, history, best_epoch, best_val_f1)
            print(f"\nMiglior modello salvato in: {self.config.best_model_path}")
            print(f"Miglior epoca: {best_epoch} | Miglior Val F1 Macro: {best_val_f1:.4f}")

        return self.build_result(history, best_epoch, best_val_f1)

    # ----------------------------------------------------------
    # Checkpoint e risultato
    # ----------------------------------------------------------
    def config_signature(self):
        """
        Configurazione dell'esperimento in forma di dizionario, usata per verificare
        che un checkpoint sia stato prodotto con gli stessi parametri e gli stessi file
        accettati. Sono esclusi i campi che non cambiano il training.
        """
        signature = asdict(self.config)
        signature.pop("best_model_path")
        signature.pop("resume_from_checkpoint")
        # img_size è stato aggiunto dopo i primi sette esperimenti: quando vale None
        # (risoluzione nativa) non entra nella firma, così i checkpoint già salvati
        # restano validi e non vengono riaddestrati.
        if signature["img_size"] is None:
            signature.pop("img_size")
        return signature

    def save_checkpoint(self, best_state, history, best_epoch, best_val_f1):
        """
        Salva in un unico file i pesi migliori, lo storico delle metriche e la
        configurazione. Il file viene scritto prima in una copia temporanea e poi
        rinominato: un'interruzione durante il salvataggio non lascia un checkpoint corrotto.
        I valori numerici sono convertiti in float Python per poterli ricaricare
        con torch.load(weights_only=True).
        """
        checkpoint = {
            "model_state": best_state,
            "history": {key: [float(v) for v in values] for key, values in history.items()},
            "best_epoch": int(best_epoch),
            "best_val_f1": float(best_val_f1),
            "config": self.config_signature(),
        }
        tmp_path = self.config.best_model_path + ".tmp"
        torch.save(checkpoint, tmp_path)
        os.replace(tmp_path, self.config.best_model_path)

    def load_checkpoint(self):
        """
        Restituisce il checkpoint salvato se la ripresa è attiva, il file esiste e la
        configurazione coincide con quella corrente; altrimenti None (nuovo training).
        """
        path = self.config.best_model_path
        if not self.config.resume_from_checkpoint or not os.path.isfile(path):
            return None
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        if checkpoint.get("config") != self.config_signature():
            print(f"[INFO] Il checkpoint {path} ha una configurazione diversa: nuovo training.")
            return None
        return checkpoint

    def build_result(self, history, best_epoch, best_val_f1):
        """Dizionario restituito da run(), uguale per un nuovo training e per una ripresa."""
        return {
            "model": self.model, 
            "history": history, 
            "criterion": self.eval_criterion,
            "train_loader": self.train_loader, 
            "val_loader": self.val_loader,
            "test_loader": self.test_loader, 
            "class_names": self.class_names,
            "best_epoch": best_epoch, 
            "best_val_f1": best_val_f1,
            "data_cfg": self.data_cfg,
        }


## 6. Classe `Evaluator`: valutazione e visualizzazione dei risultati

La fase di valutazione viene separata dal training attraverso la classe Evaluator, così da utilizzare la stessa procedura per analizzare e confrontare i diversi esperimenti sul validation set e, solo dopo la selezione del modello finale, effettuare una singola valutazione sul test set.

Oltre alle metriche principali, viene generata la matrice di confusione normalizzata, utile per osservare il comportamento del modello sulle singole classi. Viene inoltre prodotta una rappresentazione delle coppie di classi più frequentemente confuse, che permette di individuare più facilmente gli errori ricorrenti senza dover analizzare l'intera matrice di confusione.

In [ ]:

# ============================================================
# CELLA 10 — Evaluator: valutazione e visualizzazione dei risultati
# ============================================================
# Separare la logica di valutazione dall'ExperimentRunner permette di:
# 1. Valutare qualsiasi modello su qualsiasi loader (val o test) indipendentemente
# 2. Confrontare esperimenti diversi con la stessa interfaccia
# 3. Mantenere il codice di visualizzazione separato da quello di training
# ============================================================

class Evaluator:
    def __init__(self, model, loader, criterion, device, class_names, split_name="validation", use_amp=True):
        self.model = model
        self.loader = loader
        self.criterion = criterion
        self.device = device
        self.class_names = class_names
        self.display_names = [DISPLAY_NAME.get(c, c) for c in class_names]
        self.split_name = split_name
        self.use_amp = use_amp

    def plot_history(self, history):
        """
        Visualizza l'andamento di loss e F1-score macro durante il training.

        Due grafici affiancati:
        - Sinistra: loss di training vs validation (ideale: entrambe decrescenti)
        - Destra: F1 macro di training vs validation (ideale: entrambi crescenti)

        Pattern da cercare:
        - Overfitting: train_loss scende, val_loss risale → modello memorizza i dati
        - Underfitting: entrambe le loss restano alte → modello troppo semplice
        - Buon fit: entrambe le curve convergono a valori bassi

        Con Mixup il F1 di training non viene calcolato (vale NaN): nel grafico
        a destra compare quindi solo la curva di validation. Anche la loss di
        training non è direttamente confrontabile con quella di validation quando
        si usano label smoothing o Mixup.
        """
        epochs = range(1, len(history["train_loss"]) + 1)
        plt.figure(figsize=(12, 4))

        plt.subplot(1, 2, 1)
        plt.plot(epochs, history["train_loss"], label="train_loss")
        plt.plot(epochs, history["val_loss"], label="val_loss")
        plt.xlabel("Epoca"); plt.ylabel("Loss"); plt.title("Andamento della loss"); plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(epochs, history["train_f1"], label="train_f1_macro")
        plt.plot(epochs, history["val_f1"], label="val_f1_macro")
        plt.xlabel("Epoca"); plt.ylabel("F1 Macro"); plt.title("Andamento del F1 Macro"); plt.legend()

        plt.tight_layout()
        plt.show()
        plt.close()

    @torch.no_grad()
    def evaluate(self):
        """
        Esegue la valutazione completa del modello sul loader specificato.

        @torch.no_grad(): disabilita il calcolo del gradiente → più veloce e
        meno memoria. Non serve la backpropagation in fase di valutazione.

        Restituisce un dizionario con:
        - loss: cross-entropy media su tutti i campioni
        - accuracy: quota di predizioni corrette
        - f1_macro: F1-score medio tra le classi (metrica principale)
        - f1_weighted: F1 medio pesato per il numero di immagini di ogni classe
        - top3_accuracy: quota di immagini con la classe corretta tra le 3 più probabili
        - y_true, y_pred: etichette reali e predette (per classification report e matrice)
        - y_proba: probabilità softmax per classe (usate per la top-3)
        """
        self.model.eval()
        running_loss = 0.0
        y_true, y_pred, y_proba = [], [], []

        for images, labels in self.loader:
            images = images.to(self.device, non_blocking=True)
            labels = labels.to(self.device, non_blocking=True)

            with torch.autocast(device_type=self.device.type,
                                 enabled=(self.use_amp and self.device.type == "cuda")):
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs.float(), dim=1)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_proba.extend(probs.cpu().numpy())

        n_classes = len(self.class_names)
        loss_value = running_loss / len(self.loader.dataset)
        acc_value = accuracy_score(y_true, y_pred)
        f1_value = f1_score(y_true, y_pred, average="macro")
        f1_weighted = f1_score(y_true, y_pred, average="weighted")
        top3_value = top_k_accuracy_score(y_true, y_proba, k=3, labels=list(range(n_classes)))

        return {
            "loss": loss_value, 
            "accuracy": acc_value, 
            "f1_macro": f1_value,
            "f1_weighted": f1_weighted, 
            "top3_accuracy": top3_value,
            "y_true": y_true, 
            "y_pred": y_pred, 
            "y_proba": y_proba,
        }

    def plot_confusion_matrix(self, y_true, y_pred, normalize=True):
        """
        Visualizza la matrice di confusione 14x14.

        - Righe: classe reale; colonne: classe predetta.
        - Diagonale principale: predizioni corrette per ogni classe.
        - Fuori diagonale: errori; la cella (X, Y) conta le immagini di X predette come Y.

        Con 14 classi i conteggi assoluti sono poco leggibili: con normalize=True
        ogni riga viene divisa per il numero di immagini della classe reale (somma 1),
        così la cella (X, Y) si legge come "quota delle immagini della classe X
        classificate come Y". Dopo la pulizia dei duplicati le classi non hanno più
        tutte lo stesso numero di immagini, quindi la normalizzazione rende le righe
        confrontabili.

        Restituisce sempre la matrice con i conteggi assoluti, usata da
        top_confused_pairs().
        """
        cm = confusion_matrix(y_true, y_pred, labels=list(range(len(self.class_names))))
        if normalize:
            cm_display = cm.astype("float") / cm.sum(axis=1, keepdims=True)
            fmt = ".2f"
        else:
            cm_display = cm
            fmt = "d"

        fig, ax = plt.subplots(figsize=(10, 9))
        im = ax.imshow(cm_display, cmap="Blues")
        ax.set_xticks(range(len(self.display_names)))
        ax.set_yticks(range(len(self.display_names)))
        ax.set_xticklabels(self.display_names, rotation=45, ha="right")
        ax.set_yticklabels(self.display_names)
        ax.set_xlabel("Classe predetta")
        ax.set_ylabel("Classe reale")
        title = "normalizzata per riga" if normalize else "conteggi assoluti"
        ax.set_title(f"Matrice di confusione ({title}) - {self.split_name}")

        thresh = cm_display.max() / 2.0
        for i in range(cm_display.shape[0]):
            for j in range(cm_display.shape[1]):
                value = cm_display[i, j]
                if value > 0:
                    ax.text(j, i, format(value, fmt), ha="center", va="center",
                            fontsize=7, color="white" if value > thresh else "black")

        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()
        plt.close()
        return cm

    def top_confused_pairs(self, cm, top_n=10):
        """
        Individua gli errori più frequenti del modello, cioè le coppie di classi
        che vengono scambiate più spesso.

        Scorre tutte le celle fuori dalla diagonale della matrice di confusione:
        la cella (X, Y) indica quante immagini della classe reale X sono state
        predette come classe Y. Le coppie vengono ordinate per numero di errori
        e si tengono le prime top_n (es. "Taco -> Taquito: 12 immagini").

        La coppia è orientata: "Taco -> Taquito" e "Taquito -> Taco" sono due
        errori distinti e compaiono separatamente.

        Input:
        - cm: matrice di confusione con i conteggi assoluti (non normalizzata),
          come restituita da plot_confusion_matrix()
        - top_n: numero massimo di coppie da mostrare

        Output:
        - grafico a barre orizzontali delle coppie più confuse
        - DataFrame con colonne "Classe reale", "Predetta come", "N. errori"

        Rispetto alla matrice completa 14x14 mostra subito dove il modello sbaglia
        di più, per esempio tra piatti visivamente simili.
        """
        pairs = []
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                if i != j and cm[i, j] > 0:
                    pairs.append((self.display_names[i], self.display_names[j], cm[i, j]))
        pairs.sort(key=lambda x: x[2], reverse=True)
        top_pairs = pairs[:top_n]

        pairs_df = pd.DataFrame(top_pairs, columns=["Classe reale", "Predetta come", "N. errori"])

        if len(top_pairs) > 0:
            labels = [f"{r} -> {p}" for r, p, _ in top_pairs]
            values = [n for _, _, n in top_pairs]
            plt.figure(figsize=(9, max(3, 0.4 * len(labels))))
            plt.barh(labels[::-1], values[::-1], color="#c44e52")
            plt.xlabel("Numero di immagini confuse")
            plt.title(f"Le {len(top_pairs)} coppie di classi più confuse - {self.split_name}")
            plt.tight_layout()
            plt.show()
        plt.close()

        return pairs_df

    def full_report(self, history=None, show_cm=True):
        """
        Esegue la valutazione completa e produce tutti i report.

        1. Grafico loss e F1 durante il training (se history è fornito)
        2. Metriche aggregate: loss, accuracy, F1 macro, F1 weighted, top-3 accuracy
        3. Classification report per classe: precision, recall, F1, support
        4. Matrice di confusione e coppie di classi più confuse (se show_cm=True)

        Classification report:
        - Precision: su tutto ciò che il modello predice come classe X,
          quanto spesso ha ragione? (TP / (TP + FP))
        - Recall: su tutti i campioni reali della classe X,
          quanti ne trova il modello? (TP / (TP + FN))
        - F1: media armonica di precision e recall
        - Support: numero di campioni reali per classe
        """
        if history is not None:
            self.plot_history(history)

        metrics = self.evaluate()

        print("\n" + "=" * 70)
        print(f"RISULTATI SU {self.split_name.upper()}")
        print("=" * 70)
        print(f"Loss:            {metrics['loss']:.4f}")
        print(f"Accuracy:        {metrics['accuracy']:.4f}")
        print(f"F1 Macro:        {metrics['f1_macro']:.4f}")
        print(f"F1 Weighted:     {metrics['f1_weighted']:.4f}")
        print(f"Top-3 Accuracy:  {metrics['top3_accuracy']:.4f}")

        print("\nClassification Report:")
        print(classification_report(
            metrics["y_true"], metrics["y_pred"],
            labels=list(range(len(self.class_names))),
            target_names=self.display_names, digits=4, zero_division=0
        ))

        if show_cm:
            cm = self.plot_confusion_matrix(metrics["y_true"], metrics["y_pred"])
            confused_df = self.top_confused_pairs(cm, top_n=10)
            print("\nCoppie di classi più confuse:")
            print(confused_df.to_string(index=False))
            metrics["top_confused_pairs"] = confused_df

        return metrics


## 7. Configurazione degli esperimenti

Vengono definite diverse configurazioni sperimentali per confrontare progressivamente le principali scelte di training. Tutti gli esperimenti utilizzano lo stesso **dataset pulito** e mantengono comuni, salvo dove esplicitamente indicato, i principali parametri di addestramento, in modo da rendere i confronti il più possibile interpretabili.

I primi **7 esperimenti** costituiscono il confronto principale:

| # | Esperimento | Confronto | Modifica principale |
|---|---|---|---|
| 1 | `baseline_basic` | riferimento | backbone congelato e augmentation basic |
| 2 | `frozen_moderate` | 2 vs 1 | augmentation basic → moderate |
| 3 | `progressive_moderate` | 3 vs 2 | sblocco completo del backbone dall'epoca 4 |
| 4 | `full_ft_moderate` | 4 vs 2 | backbone allenabile fin dalla prima epoca |
| 5 | `full_ft_smoothing` | 5 vs 4 | introduzione del label smoothing = 0.1 |
| 6 | `strong_aug_label_smoothing` | 6 vs 5 | augmentation moderate → strong |
| 7 | `strong_aug_mixup` | 7 vs 6 | introduzione del Mixup con α = 0.2 |

Questa sequenza permette di partire da una **baseline semplice**, nella quale viene allenata soltanto la testa finale del modello, e valutare progressivamente l'effetto dell'augmentation, delle diverse strategie di fine-tuning e delle tecniche di regolarizzazione.

Il confronto tra gli esperimenti **3 e 4** valuta due diverse strategie di fine-tuning. Nell'esperimento 3 il backbone viene inizialmente mantenuto congelato e sbloccato dall'epoca 4, mentre nell'esperimento 4 è allenabile fin dalla prima epoca.

### Esperimenti di approfondimento

Vengono inoltre definiti **due esperimenti di approfondimento**, costruiti a partire dalla configurazione dell'esperimento 6:

| # | Esperimento | Confronto | Modifica principale |
|---|---|---|---|
| 8 | `strong_aug_label_smoothing_288px` | 8 vs 6 | dimensione input 224×224 → 288×288 |
| 9 | `strong_aug_label_smoothing_288px_b2` | 9 vs 8 | EfficientNet-B0 → EfficientNet-B2 |

L'esperimento 8 verifica se una **risoluzione maggiore** permette di conservare più dettagli visivi utili a distinguere classi simili.

L'esperimento 9 mantiene la risoluzione a **288×288** e sostituisce EfficientNet-B0 con **EfficientNet-B2**, per verificare se una maggiore capacità del modello permette di migliorare ulteriormente la distinzione tra le classi più difficili.

Gli esperimenti 8 e 9 rappresentano quindi **approfondimenti successivi basati sui risultati e sugli errori osservati negli esperimenti precedenti**, senza modificare la struttura del confronto principale.


In [ ]:
# ============================================================
# CELLA 11 — Configurazione degli esperimenti
# ============================================================
# Le configurazioni sono esplicite, per leggere facilmente cosa cambia in ogni test.
# Manteniamo comuni seed, learning rate iniziale, batch size e budget massimo.
# ============================================================

if not globals().get("accepted_paths"):
    raise ValueError("Eseguire prima la cella 5.5 per applicare il piano di pulizia.")
base_cfg = Config(allowed_paths=accepted_paths)

# Esperimento 1 — Baseline: backbone congelato, augmentation basic
baseline_cfg = copy.deepcopy(base_cfg)
baseline_cfg.experiment_name = "baseline_basic"
baseline_cfg.best_model_path = "best_baseline_basic.pth"

# Esperimento 2 — Backbone congelato, augmentation moderata
frozen_cfg = copy.deepcopy(baseline_cfg)
frozen_cfg.experiment_name = "frozen_moderate"
frozen_cfg.best_model_path = "best_frozen_moderate.pth"
frozen_cfg.augmentation = 'moderate'

# Esperimento 3 — Fine-tuning in due fasi
progressive_cfg = copy.deepcopy(frozen_cfg)
progressive_cfg.experiment_name = "progressive_moderate"
progressive_cfg.best_model_path = "best_progressive_moderate.pth"
progressive_cfg.progressive_unfreeze = True
progressive_cfg.unfreeze_epoch = 4

# Esperimento 4 — Fine-tuning completo fin dalla prima epoca
full_ft_cfg = copy.deepcopy(frozen_cfg)
full_ft_cfg.experiment_name = "full_ft_moderate"
full_ft_cfg.best_model_path = "best_full_ft_moderate.pth"
full_ft_cfg.freeze_backbone = False

# Esperimento 5 — Fine-tuning completo e label smoothing
smoothing_cfg = copy.deepcopy(full_ft_cfg)
smoothing_cfg.experiment_name = "full_ft_smoothing"
smoothing_cfg.best_model_path = "best_full_ft_smoothing.pth"
smoothing_cfg.label_smoothing = 0.1

# Esperimento 6 — Augmentation forte e label smoothing
strong_aug_cfg = copy.deepcopy(smoothing_cfg)
strong_aug_cfg.experiment_name = "strong_aug_label_smoothing"
strong_aug_cfg.best_model_path = "best_strong_aug_label_smoothing.pth"
strong_aug_cfg.augmentation = 'strong'

# Esperimento 7 — Augmentation forte e Mixup
mixup_cfg = copy.deepcopy(strong_aug_cfg)
mixup_cfg.experiment_name = "strong_aug_mixup"
mixup_cfg.best_model_path = "best_strong_aug_mixup.pth"
mixup_cfg.mixup_alpha = 0.2

# ------------------------------------------------------------
# Esperimenti di approfondimento sulla configurazione dell'esperimento 6
# ------------------------------------------------------------
# Esperimento 8 — Come l'esperimento 6, con immagini 288x288 invece di 224x224.
# Gli errori residui riguardano dettagli fini (Taco/Taquito, Apple Pie/Cheesecake/Ice Cream):
# una risoluzione più alta conserva più dettagli. Manteniamo EfficientNet-B0, più leggero
# di B2/B3: il costo di calcolo cresce di circa 1.65 volte (288² / 224²).
highres_cfg = copy.deepcopy(strong_aug_cfg)
highres_cfg.experiment_name = "strong_aug_label_smoothing_288px"
highres_cfg.best_model_path = "best_strong_aug_label_smoothing_288px.pth"
highres_cfg.img_size = 288

# Esperimento 9 — Come l'esperimento 8, con EfficientNet-B2 al posto di EfficientNet-B0.
# La risoluzione più alta ha aiutato i dessert ma non Taco/Taquito: verifichiamo se un
# modello più grande distingue meglio i piatti simili. Cambia solo l'architettura;
# mean, std e preprocessing restano quelli del pretraining su ImageNet.
# Il costo di calcolo per immagine è circa il doppio dell'esperimento 8.
b2_cfg = copy.deepcopy(highres_cfg)
b2_cfg.experiment_name = "strong_aug_label_smoothing_288px_b2"
b2_cfg.best_model_path = "best_strong_aug_label_smoothing_288px_b2.pth"
b2_cfg.model_name = "efficientnet_b2"

experiments = [baseline_cfg, frozen_cfg, progressive_cfg, full_ft_cfg, smoothing_cfg, strong_aug_cfg, mixup_cfg,
               highres_cfg, b2_cfg]

# I checkpoint vengono salvati nella cartella persistente definita nella cella 1B (Google Drive).
for cfg in experiments:
    cfg.best_model_path = os.path.join(CHECKPOINT_DIR, cfg.best_model_path)

print(f"Definiti {len(experiments)} esperimenti:")
for cfg in experiments:
    print(f"  - {cfg.experiment_name} -> {cfg.best_model_path}")


In [ ]:
# ============================================================
# TEST 1 — Baseline: backbone congelato, augmentation basic
# ============================================================
# Alleniamo soltanto la testa finale. È il riferimento iniziale per tutti i confronti.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(baseline_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

baseline_metrics = evaluator.full_report(history=result["history"])
baseline_result = result
baseline_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
baseline_result["model"].to("cpu")
del runner

### Esperimento 1 — Baseline: backbone congelato + augmentation basic

Il primo esperimento rappresenta la **baseline**, cioè il punto di riferimento con cui confrontare le configurazioni successive.

Viene utilizzata **EfficientNet-B0 pre-addestrata su ImageNet**, mantenendo congelato il backbone e allenando solamente la nuova testa di classificazione sulle 14 classi del dataset. Viene inoltre applicato il livello di augmentation `basic`.

L'obiettivo è verificare quali prestazioni è possibile ottenere sfruttando le feature già apprese su ImageNet, prima di introdurre augmentation più intensa, fine-tuning del backbone o ulteriori tecniche di regolarizzazione.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 19 (early stopping all'epoca 23) |
| Loss | 0.4700 |
| Accuracy | 0.8537 |
| F1 Macro | 0.8547 |
| F1 Weighted | 0.8547 |
| Top-3 Accuracy | 0.9670 |

#### Analisi dei risultati

La baseline raggiunge un **F1 Macro di 0.8547** e un'**Accuracy dell'85.37%** sul validation set. F1 Macro e F1 Weighted coincidono perché il validation set è quasi bilanciato (da 151 a 160 immagini per classe): in questo caso le due medie danno praticamente lo stesso risultato.

La **Top-3 Accuracy del 96.70%** mostra che, anche quando la prima previsione è errata, nella grande maggioranza dei casi la classe corretta è comunque presente tra le prime tre classi considerate più probabili dal modello.

Le curve mostrano un apprendimento rapido nelle prime epoche e poi un **plateau**. Il F1 Macro di validation sale da **0.6099** alla prima epoca a **0.8502** all'undicesima; nelle otto epoche successive guadagna meno di mezzo punto, fino al massimo di **0.8547 all'epoca 19**. Dopo quattro epoche senza miglioramenti interviene l'**early stopping** all'epoca 23, con il checkpoint dell'epoca 19.

La validation loss continua invece a scendere lentamente fino alla fine (da **1.2450** a **0.4627**): il modello diventa un po' più sicuro delle predizioni corrette, ma senza cambiare in modo apprezzabile le classi predette.

Durante il training cresce il **divario tra training e validation**: all'epoca 19 il F1 Macro è **0.9385 sul training** contro **0.8547 sulla validation**, mentre alla quinta epoca i due valori erano ancora vicini (**0.8348** e **0.8136**). La sola testa di classificazione continua ad adattarsi ai dati di training, ma le feature fisse di ImageNet limitano il miglioramento sulla validation.

#### Analisi delle classi

Le prestazioni non sono identiche per tutte le classi. Gli F1-score più bassi si osservano per:

- `Taco`: **0.7251**
- `Apple Pie`: **0.7692**
- `Taquito`: **0.7973**
- `Cheesecake`: **0.8061**

Le classi riconosciute meglio sono invece `Donut` (**0.9385**), `Fries` (**0.9109**), `Chicken Curry` (**0.9080**) e `Sushi` (**0.9068**).

`Taco` ha la **precision più bassa** (**0.6703**): circa un'immagine su tre predetta come Taco appartiene in realtà a un'altra classe. La colonna `Taco` della matrice di confusione mostra infatti che il modello vi attribuisce immagini di diverse classi, soprattutto `Taquito` (13%), `Sandwich` (6%) e `Crispy Chicken` (5%). `Taquito` ha invece il **recall più basso** (**0.7389**): circa un quarto delle sue immagini viene assegnato ad altre classi.

Le confusioni più frequenti sono:

- `Taquito → Taco`: **21 immagini**
- `Cheesecake → Apple Pie`: **19 immagini**
- `Apple Pie → Cheesecake`: **17 immagini**
- `Ice Cream → Cheesecake`: **12 immagini**
- `Sandwich → Taco` e `Taco → Sandwich`: **9 immagini** ciascuna

Emergono quindi due gruppi di classi difficili da separare: **Taco/Taquito**, spesso confusi anche con `Sandwich`, e i dessert **Apple Pie/Cheesecake/Ice Cream**, con una confusione quasi simmetrica tra Apple Pie e Cheesecake.

#### Conclusione

La baseline fornisce già prestazioni soddisfacenti pur allenando solamente la testa finale di classificazione. Tuttavia, il plateau del F1 di validation dopo l'undicesima epoca, il divario crescente con il training e le confusioni tra classi visivamente simili indicano che le feature fisse di ImageNet non bastano a separare alcune categorie.

Nel prossimo esperimento viene quindi mantenuto il **backbone congelato**, modificando solamente l'augmentation da `basic` a `moderate`. In questo modo sarà possibile verificare se una maggiore variabilità delle immagini di training migliora la capacità di generalizzazione del modello.

In [ ]:
# ============================================================
# TEST 2 — Backbone congelato, augmentation moderata
# ============================================================
# Confronto con il test 1: cambia soltanto il pacchetto di augmentation, da basic a moderate.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(frozen_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

frozen_metrics = evaluator.full_report(history=result["history"])
frozen_result = result
frozen_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
frozen_result["model"].to("cpu")
del runner

### Esperimento 2 — Backbone congelato + augmentation moderate

Nel secondo esperimento manteniamo la stessa strategia della baseline: **EfficientNet-B0 pre-addestrata su ImageNet con backbone congelato**, allenando solamente la testa finale di classificazione.

L'unica modifica riguarda l'augmentation, che passa da `basic` a `moderate`.

L'obiettivo è verificare se una maggiore variabilità delle immagini durante il training permette di migliorare la generalizzazione del modello senza modificare l'architettura o la strategia di fine-tuning.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 25 (ultima epoca disponibile) |
| Loss | 0.4341 |
| Accuracy | 0.8645 |
| F1 Macro | 0.8654 |
| F1 Weighted | 0.8654 |
| Top-3 Accuracy | 0.9670 |

#### Analisi dei risultati

Con l'augmentation `moderate` il modello raggiunge un **F1 Macro di 0.8654** e un'**Accuracy dell'86.45%** sul validation set. La **Top-3 Accuracy è del 96.70%**, identica alla baseline.

Le curve mostrano un apprendimento regolare. Il F1 Macro di validation sale da **0.5917** alla prima epoca a **0.8489** all'undicesima e poi continua a crescere lentamente ma in modo costante, fino a **0.8654 all'epoca 25**. Anche la validation loss scende per tutto il training, da **1.2924** a **0.4341**, con il valore minimo all'ultima epoca.

Il miglior risultato coincide quindi con l'**ultima epoca disponibile**: l'early stopping non interviene e il training si ferma per il limite di 25 epoche. Con l'augmentation `moderate` il modello converge più lentamente rispetto alla baseline, che si era fermata all'epoca 23 con il miglior valore all'epoca 19, e potrebbe migliorare ancora di poco con più epoche.

Il divario tra training e validation resta contenuto: all'epoca 25 il F1 Macro è **0.8906 sul training** e **0.8654 sulla validation**, contro **0.9385** e **0.8547** della baseline alla sua epoca migliore. Nelle prime dodici epoche il F1 di validation è addirittura superiore a quello di training. Il confronto va però letto con cautela: il F1 di training è calcolato sulle immagini già trasformate dall'augmentation, più difficili con `moderate` che con `basic`, mentre la validation usa immagini senza trasformazioni casuali. I due valori di training dei due esperimenti non misurano quindi esattamente la stessa cosa.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Taco`: **0.7552**
- `Apple Pie`: **0.7664**
- `Cheesecake`: **0.8085**
- `Taquito`: **0.8133**

Le classi riconosciute meglio sono `Donut` (**0.9389**), `Chicken Curry` (**0.9226**), `Fries` (**0.9195**) e `Sushi` (**0.9137**).

Rispetto alla baseline migliorano soprattutto **`Taco`** (da 0.7251 a 0.7552) e **`Taquito`** (da 0.7973 a 0.8133), mentre `Apple Pie` e `Cheesecake` restano praticamente invariate. `Taco` rimane la classe con la precision più bassa (**0.7033**) e `Taquito` quella con il recall più basso (**0.7771**).

Le principali confusioni sono:

- `Cheesecake → Apple Pie`: **20 immagini**
- `Taquito → Taco`: **19 immagini**
- `Apple Pie → Cheesecake`: **16 immagini**
- `Ice Cream → Cheesecake`: **12 immagini**
- `Ice Cream → Apple Pie`: **9 immagini**

Rimangono quindi difficili da separare soprattutto i dessert **Apple Pie/Cheesecake/Ice Cream**, ora al primo posto tra le confusioni, e la coppia **Taco/Taquito**.

#### Confronto con la baseline

Rispetto all'esperimento 1 cambia soltanto il livello di augmentation:

| Metrica | Baseline | Moderate | Differenza |
|---|---:|---:|---:|
| Loss | 0.4700 | 0.4341 | -0.0359 |
| Accuracy | 0.8537 | 0.8645 | +0.0108 |
| F1 Macro | 0.8547 | 0.8654 | +0.0107 |
| Top-3 Accuracy | 0.9670 | 0.9670 | 0.0000 |

L'augmentation `moderate` migliora il F1 Macro di circa **1.1 punti percentuali** e riduce il divario tra training e validation. Nel primo run con budget di 15 epoche lo stesso confronto dava solo +0.4 punti: l'augmentation più intensa rende il training più lento, e il vantaggio emerge con un budget di epoche più ampio.

L'augmentation `moderate` sembra quindi agire come forma di **regolarizzazione**: rende il compito di training più difficile, ma migliora le prestazioni sulla validation. Il miglioramento riguarda soprattutto Taco e Taquito, non i dessert.

#### Conclusione

L'esperimento 2 mostra che passare da augmentation `basic` a `moderate` porta a un **miglioramento di circa un punto di F1 Macro** e a un divario più contenuto tra training e validation, mantenendo il backbone congelato.

L'augmentation `moderate` viene quindi mantenuta nell'esperimento successivo, nel quale verrà modificata la strategia di fine-tuning introducendo lo **sblocco del backbone dall'epoca 4**.

In [ ]:
# ============================================================
# TEST 3 — Fine-tuning in due fasi
# ============================================================
# Confronto con il test 2: dopo tre epoche sblocchiamo tutto il backbone. Ottimizzatore e scheduler vengono ricreati allo sblocco.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(progressive_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

progressive_metrics = evaluator.full_report(history=result["history"])
progressive_result = result
progressive_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
progressive_result["model"].to("cpu")
del runner

### Esperimento 3 — Fine-tuning progressivo + augmentation moderate

Nel terzo esperimento manteniamo l'augmentation `moderate` utilizzata nell'esperimento precedente, ma modifichiamo la strategia di addestramento del modello.

Durante le prime **3 epoche** il backbone di EfficientNet-B0 rimane congelato e viene allenata solamente la testa finale di classificazione. A partire dall'**epoca 4**, il backbone viene completamente sbloccato, permettendo di aggiornare anche i pesi pre-addestrati su ImageNet.

L'obiettivo è verificare se una prima fase con backbone congelato, seguita dal fine-tuning dell'intera rete, permette al modello di adattare meglio le feature al dataset di immagini alimentari.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 23 (training completato fino all'epoca 25) |
| Loss | 0.4603 |
| Accuracy | 0.9092 |
| F1 Macro | 0.9092 |
| F1 Weighted | 0.9092 |
| Top-3 Accuracy | 0.9706 |

#### Analisi dei risultati

Lo sblocco del backbone produce un miglioramento evidente rispetto all'esperimento precedente. Il miglior risultato viene raggiunto all'**epoca 23**, con un **F1 Macro di 0.9092**, un'**Accuracy del 90.92%** e una **Top-3 Accuracy del 97.06%**.

Le prime tre epoche, con backbone congelato, sono identiche a quelle dell'esperimento 2 (F1 di validation **0.7677** alla terza epoca): a parità di seed e configurazione il training è riproducibile. Allo sblocco del backbone, all'epoca 4, il F1 Macro di validation sale subito a **0.8518** e all'epoca 5 a **0.8727**, già oltre il miglior risultato ottenuto con il backbone sempre congelato (0.8654).

Tra l'epoca 5 e l'epoca 18 il F1 di validation oscilla tra **0.857 e 0.889** senza una crescita netta, mentre la validation loss risale da **0.4447** a **0.6430** (epoca 17) e il training continua a migliorare: sono i primi segnali di adattamento eccessivo ai dati di training. Dopo le epoche 16-18 senza miglioramenti, lo scheduler `ReduceLROnPlateau` dimezza il learning rate: all'epoca 19 il F1 di validation sale di colpo a **0.9076** e la validation loss scende a **0.4335**, il valore minimo del training. L'early stopping non interviene, perché il miglioramento arriva dopo tre epoche senza progressi, una prima della soglia di quattro.

Il divario tra training e validation è ampio: alla migliore epoca il F1 Macro è **0.9840 sul training** e **0.9092 sulla validation**, con una training loss di appena **0.0529**. Il modello si adatta quasi perfettamente ai dati di training, e il checkpoint dell'epoca 23 conserva il punto migliore sulla validation.

#### Analisi delle classi

Tutte le classi superano un F1-score di **0.85**. I valori più bassi si osservano per:

- `Taco`: **0.8562**
- `Sandwich`: **0.8758**
- `Apple Pie`: **0.8758**
- `Taquito`: **0.8874**

Le classi riconosciute meglio sono `Donut` (**0.9627**), `Baked Potato` (**0.9497**), `Chicken Curry` (**0.9404**) e `Fries` (**0.9329**).

Il fine-tuning migliora in particolare le classi più difficili degli esperimenti precedenti: rispetto all'esperimento 2, `Apple Pie` passa da 0.7664 a **0.8758**, `Taco` da 0.7552 a **0.8562** e `Cheesecake` da 0.8085 a **0.8952**.

Le principali confusioni sono:

- `Taquito → Taco`: **13 immagini**
- `Cheesecake → Ice Cream`: **9 immagini**
- `Omelette → Apple Pie`: **9 immagini**
- `Hot Dog → Sandwich`, `Sandwich → Crispy Chicken`, `Taco → Crispy Chicken`, `Cheesecake → Apple Pie` e `Ice Cream → Cheesecake`: **6 immagini** ciascuna

Gli errori sono ora più distribuiti tra le classi. La confusione `Cheesecake → Apple Pie`, la più frequente nell'esperimento 2 con 20 immagini, scende a 6; `Taquito → Taco` rimane invece l'errore più frequente.

#### Confronto con l'esperimento precedente

Rispetto all'esperimento 2, l'augmentation rimane `moderate`: ciò che cambia è lo **sblocco del backbone dall'epoca 4**.

| Metrica | Backbone congelato | Fine-tuning progressivo | Differenza |
|---|---:|---:|---:|
| Loss | 0.4341 | 0.4603 | +0.0262 |
| Accuracy | 0.8645 | 0.9092 | +0.0447 |
| F1 Macro | 0.8654 | 0.9092 | +0.0438 |
| Top-3 Accuracy | 0.9670 | 0.9706 | +0.0036 |

Il F1 Macro aumenta di circa **4.4 punti percentuali**, un vantaggio molto più marcato di quello ottenuto modificando solamente l'augmentation.

La validation loss è invece leggermente più alta, nonostante le predizioni corrette siano molte di più. La loss penalizza anche la sicurezza con cui il modello sbaglia: dopo il fine-tuning il modello è molto sicuro delle proprie predizioni, e i pochi errori rimasti pesano di più sulla loss. Per scegliere il modello conta il F1 Macro, che misura direttamente la correttezza delle classi predette.

#### Conclusione

Il fine-tuning del backbone porta a un miglioramento importante delle prestazioni: il modello non utilizza più solamente le feature generiche apprese su ImageNet, ma può adattarle alle caratteristiche specifiche delle immagini alimentari.

Lo sblocco progressivo raggiunge un **F1 Macro di 0.9092**, rispetto a **0.8654** del modello con backbone sempre congelato. Il divario ampio con il training e l'andamento della validation loss indicano però che serve regolarizzazione.

Nel prossimo esperimento verifichiamo se sia realmente necessario attendere tre epoche prima di sbloccare il backbone oppure se sia più efficace eseguire il **fine-tuning completo fin dalla prima epoca**.

In [ ]:
# ============================================================
# TEST 4 — Fine-tuning completo fin dalla prima epoca
# ============================================================
# Confronto con il test 2: rendiamo allenabile tutto il backbone fin dall’inizio. Il confronto con il test 3 riguarda invece le due strategie di fine-tuning.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(full_ft_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

full_ft_metrics = evaluator.full_report(history=result["history"])
full_ft_result = result
full_ft_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
full_ft_result["model"].to("cpu")
del runner

### Esperimento 4 — Fine-tuning completo + augmentation moderate

Nel quarto esperimento manteniamo l'augmentation `moderate`, ma rendiamo **l'intera EfficientNet-B0 allenabile fin dalla prima epoca**.

A differenza dell'esperimento precedente, quindi, non è prevista una fase iniziale con backbone congelato.

L'obiettivo è confrontare direttamente le due strategie di fine-tuning e verificare se sia più efficace adattare l'intera rete fin dall'inizio oppure procedere con uno sblocco successivo.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 24 (training completato fino all'epoca 25) |
| Loss | 0.4237 |
| Accuracy | 0.9128 |
| F1 Macro | 0.9128 |
| F1 Weighted | 0.9129 |
| Top-3 Accuracy | 0.9747 |

#### Analisi dei risultati

Il fine-tuning completo permette al modello di raggiungere rapidamente buone prestazioni. Già alla prima epoca il F1 Macro di validation è **0.8336** e alla terza (**0.8651**) eguaglia il miglior risultato ottenuto con il backbone sempre congelato nell'esperimento 2 (0.8654).

Nelle prime dieci epoche il F1 di validation oscilla tra **0.845 e 0.878**, mentre la validation loss tende a risalire (fino a **0.5883** all'epoca 10). Dopo tre epoche senza miglioramenti, lo scheduler `ReduceLROnPlateau` dimezza il learning rate: all'epoca 11 il F1 sale a **0.8998** e la validation loss scende a **0.4093**, il valore minimo del training. Lo stesso comportamento era stato osservato nell'esperimento 3: la riduzione del learning rate è il passaggio che sblocca il miglioramento.

Nelle epoche successive il F1 di validation cresce lentamente fino a **0.9126 all'epoca 20** e poi resta sostanzialmente stabile. Il valore migliore, **0.9128 all'epoca 24**, supera quello dell'epoca 20 di appena 0.0002: il modello ha raggiunto un **plateau** già dall'epoca 20. L'early stopping non interviene perché questo piccolo miglioramento arriva dopo tre epoche senza progressi, una prima della soglia di quattro.

Anche in questo caso il divario tra training e validation è ampio: alla migliore epoca il F1 Macro è **0.9887 sul training** e **0.9128 sulla validation**, con una training loss di **0.0374**. Il modello si adatta quasi perfettamente ai dati di training.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Apple Pie`: **0.8640**
- `Sandwich`: **0.8831**
- `Taquito`: **0.8831**
- `Cheesecake`: **0.8862**

Le classi riconosciute meglio sono `Donut` (**0.9653**), `Baked Potato` (**0.9503**), `Sushi` (**0.9490**) e `Hot Dog` (**0.9299**).

Rispetto all'esperimento 3 migliora soprattutto **`Taco`**, che passa da 0.8562 a **0.8981**: la confusione `Taquito → Taco` scende da 13 a 7 immagini. `Apple Pie` diventa invece la classe più difficile, con la **precision più bassa** (**0.8266**): il modello le attribuisce immagini di altri dessert e di `Omelette`.

Le principali confusioni sono:

- `Cheesecake → Apple Pie`: **12 immagini**
- `Ice Cream → Cheesecake`: **9 immagini**
- `Omelette → Apple Pie`: **8 immagini**
- `Taquito → Taco` e `Apple Pie → Cheesecake`: **7 immagini** ciascuna
- `Omelette → Chicken Curry`: **6 immagini**

La principale area di errore sono ora i **dessert** (`Apple Pie`, `Cheesecake`, `Ice Cream`), a cui si aggiunge `Omelette`, confusa con `Apple Pie` e `Chicken Curry`.

#### Confronto con l'esperimento precedente

Il confronto con l'esperimento 3 permette di valutare direttamente le due strategie di fine-tuning.

| Metrica | Fine-tuning progressivo | Fine-tuning completo | Differenza |
|---|---:|---:|---:|
| Loss | 0.4603 | 0.4237 | -0.0366 |
| Accuracy | 0.9092 | 0.9128 | +0.0036 |
| F1 Macro | 0.9092 | 0.9128 | +0.0036 |
| Top-3 Accuracy | 0.9706 | 0.9747 | +0.0041 |

Il fine-tuning completo ottiene risultati leggermente migliori su tutte le metriche, con una validation loss più bassa. La differenza di F1 Macro è però di soli **0.36 punti**, pari a circa **8 immagini** su 2.214.

Rispetto all'esperimento 2, con backbone congelato, il fine-tuning completo guadagna invece **4.7 punti** di F1 Macro, confermando che il salto principale dipende dall'aggiornamento del backbone.

#### Conclusione

Il fine-tuning completo fin dalla prima epoca raggiunge un **F1 Macro di 0.9128**, di poco superiore allo sblocco progressivo (0.9092). In questo run non emerge quindi un vantaggio nello svolgere una fase iniziale con backbone congelato, e il fine-tuning completo raggiunge buone prestazioni già dalle prime epoche.

Per gli esperimenti successivi viene quindi mantenuto il **backbone completamente allenabile fin dall'inizio**.

Rimane però evidente un ampio divario tra le prestazioni di training e validation. Nel prossimo esperimento introduciamo quindi il **Label Smoothing**, mantenendo invariati fine-tuning completo e augmentation `moderate`, per verificare se questa forma di regolarizzazione migliora la generalizzazione.

In [ ]:
# ============================================================
# TEST 5 — Fine-tuning completo e label smoothing
# ============================================================
# Confronto con il test 4: aggiungiamo soltanto label smoothing 0.1, mantenendo augmentation moderata.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(smoothing_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

smoothing_metrics = evaluator.full_report(history=result["history"])
smoothing_result = result
smoothing_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
smoothing_result["model"].to("cpu")
del runner

### Esperimento 5 — Fine-tuning completo + Label Smoothing

Nel quinto esperimento manteniamo **EfficientNet-B0 completamente allenabile fin dalla prima epoca** e l'augmentation `moderate`, come nell'esperimento precedente.

L'unica modifica introdotta è il **Label Smoothing con valore 0.1**.

Il Label Smoothing rende meno rigidi i target utilizzati durante l'addestramento: invece di concentrare tutta la probabilità sulla sola classe corretta, ne distribuisce una piccola parte anche sulle altre classi.

L'obiettivo è ridurre l'eccessiva sicurezza del modello nelle proprie predizioni e verificare se questa forma di regolarizzazione migliora le prestazioni sul validation set.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 16 (early stopping all'epoca 20) |
| Loss | 0.3780 |
| Accuracy | 0.9192 |
| F1 Macro | 0.9194 |
| F1 Weighted | 0.9194 |
| Top-3 Accuracy | 0.9774 |

#### Analisi dei risultati

Con il Label Smoothing il modello raggiunge il risultato migliore all'**epoca 16**, con un **F1 Macro di 0.9194**, un'**Accuracy del 91.92%** e una **Top-3 Accuracy del 97.74%**. Dopo quattro epoche senza miglioramenti interviene l'**early stopping** all'epoca 20: è il primo esperimento con fine-tuning in cui il training si ferma prima del limite di 25 epoche.

Il F1 Macro di validation cresce da **0.8436** alla prima epoca a **0.8924** alla quinta, poi oscilla per tre epoche. Anche qui, dopo la riduzione del learning rate da parte dello scheduler, all'epoca 9 il F1 sale a **0.9081** e poi continua a migliorare lentamente fino a **0.9194** all'epoca 16.

La **validation loss** è più bassa e più stabile rispetto agli esperimenti precedenti: resta tra **0.37 e 0.47** per tutto il training, senza la risalita osservata negli esperimenti 3 e 4 (fino a 0.59), e raggiunge **0.3724** all'epoca 20, il valore più basso ottenuto finora.

La **training loss** rimane invece intorno a **0.60** anche nelle ultime epoche. Questo comportamento è previsto: con Label Smoothing 0.1 e 14 classi il target assegna 0.907 alla classe corretta e 0.007 a ciascuna delle altre, e anche un modello che riproducesse esattamente questo target avrebbe una loss di circa **0.55**. La training loss non può quindi scendere vicino a zero come nell'esperimento 4 e non va confrontata direttamente con quella ottenuta senza smoothing.

Il divario tra training e validation si riduce leggermente: alla migliore epoca il F1 Macro è **0.9854 sul training** e **0.9194 sulla validation**, contro **0.9887** e **0.9128** nell'esperimento 4.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Taco`: **0.8606**
- `Apple Pie`: **0.8634**
- `Cheesecake`: **0.8785**
- `Taquito`: **0.8984**

Le classi riconosciute meglio sono `Donut` (**0.9686**), `Baked Potato` (**0.9585**), `Chicken Curry` (**0.9544**) e `Sushi` (**0.9460**).

Rispetto all'esperimento 4 migliorano in particolare `Sandwich` (da 0.8831 a **0.9231**), `Omelette` (da 0.8968 a **0.9226**) e `Chicken Curry` (da 0.9292 a **0.9544**). `Taco` peggiora invece da 0.8981 a **0.8606**, con una precision di **0.8208**: la confusione `Taquito → Taco` torna a **14 immagini**, dopo le 7 dell'esperimento 4 e le 13 dell'esperimento 3. Questa coppia varia molto da un esperimento all'altro ed è la più instabile del dataset.

Le principali confusioni sono:

- `Taquito → Taco`: **14 immagini**
- `Cheesecake → Apple Pie`: **11 immagini**
- `Ice Cream → Cheesecake`: **9 immagini**
- `Apple Pie → Cheesecake` e `Omelette → Apple Pie`: **7 immagini** ciascuna
- `Cheesecake → Ice Cream`: **6 immagini**

Le difficoltà restano concentrate su **Taco/Taquito** e sui dessert **Apple Pie/Cheesecake/Ice Cream**.

#### Confronto con l'esperimento precedente

Rispetto all'esperimento 4, manteniamo fine-tuning completo e augmentation `moderate` e introduciamo solamente il **Label Smoothing = 0.1**.

| Metrica | Fine-tuning completo | + Label Smoothing | Differenza |
|---|---:|---:|---:|
| Loss | 0.4237 | 0.3780 | -0.0457 |
| Accuracy | 0.9128 | 0.9192 | +0.0064 |
| F1 Macro | 0.9128 | 0.9194 | +0.0066 |
| Top-3 Accuracy | 0.9747 | 0.9774 | +0.0027 |

Il Label Smoothing migliora tutte le metriche. Il guadagno di F1 Macro è di **0.66 punti**, circa 15 immagini su 2.214: una differenza contenuta, che con un solo seed non basta da sola a dimostrare un vantaggio stabile.

Più chiaro è l'effetto sulla **validation loss**, che scende da 0.4237 a **0.3780** ed è più stabile durante tutto il training. In validation la loss è sempre la CrossEntropy standard, quindi il confronto è corretto: il modello è meno sicuro delle proprie predizioni e, quando sbaglia, sbaglia con probabilità meno estreme, che la loss penalizza meno.

#### Conclusione

In questo esperimento il **Label Smoothing migliora le prestazioni di validation** rispetto al fine-tuning completo senza smoothing: il F1 Macro passa da **0.9128 a 0.9194**, la validation loss da **0.4237 a 0.3780** e il training si stabilizza prima, fermandosi per early stopping all'epoca 20.

Il Label Smoothing viene quindi mantenuto nell'esperimento successivo.

Nel prossimo esperimento modifichiamo solamente l'augmentation, passando da `moderate` a `strong`, per verificare se una maggiore variabilità delle immagini durante il training permette di migliorare ulteriormente la generalizzazione.

In [ ]:
# ============================================================
# TEST 6 — Augmentation forte e label smoothing
# ============================================================
# Confronto con il test 5: cambia soltanto il pacchetto di augmentation, da moderate a strong.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(strong_aug_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

strong_aug_metrics = evaluator.full_report(history=result["history"])
strong_aug_result = result
strong_aug_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
strong_aug_result["model"].to("cpu")
del runner

### Esperimento 6 — Augmentation strong + Label Smoothing

Nel sesto esperimento manteniamo **EfficientNet-B0 completamente allenabile** e il **Label Smoothing = 0.1**, come nell'esperimento precedente.

L'unica modifica riguarda l'augmentation, che passa da `moderate` a `strong`.

L'obiettivo è verificare se una maggiore variabilità delle immagini di training permette di migliorare ulteriormente la capacità di generalizzazione del modello.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 20 (early stopping all'epoca 24) |
| Loss | 0.3745 |
| Accuracy | 0.9228 |
| F1 Macro | 0.9229 |
| F1 Weighted | 0.9229 |
| Top-3 Accuracy | 0.9743 |

#### Analisi dei risultati

Con l'augmentation `strong` il modello raggiunge il risultato migliore all'**epoca 20**, con un **F1 Macro di 0.9229**, un'**Accuracy del 92.28%** e una **Top-3 Accuracy del 97.43%**. Dopo quattro epoche senza miglioramenti interviene l'**early stopping** all'epoca 24.

Il F1 Macro di validation sale rapidamente da **0.8502** alla prima epoca a **0.9010** alla quinta, poi oscilla tra 0.893 e 0.902 fino all'epoca 10. Dopo ogni fase senza miglioramenti lo scheduler riduce il learning rate e il F1 riprende a crescere: **0.9137** all'epoca 13, **0.9178** alla 15 e **0.9229** alla 20. La crescita è più graduale rispetto agli esperimenti 3 e 4, dove la riduzione del learning rate produceva un salto netto.

La validation loss resta bassa e stabile per tutto il training (tra **0.37 e 0.47**) e raggiunge **0.3689** all'epoca 19, il valore più basso tra tutti gli esperimenti. Come nell'esperimento 5, la training loss resta intorno a **0.63** per effetto del Label Smoothing, ed è un po' più alta perché le immagini di training sono ora più difficili.

Il **divario tra training e validation è il più contenuto** tra gli esperimenti con fine-tuning: alla migliore epoca il F1 Macro è **0.9728 sul training** e **0.9229 sulla validation**, circa 5 punti, contro i 6.6 dell'esperimento 5 e i 7.6 dell'esperimento 4. L'augmentation `strong` rende più difficile memorizzare le immagini di training.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Apple Pie`: **0.8726**
- `Taco`: **0.8742**
- `Cheesecake`: **0.8862**
- `Taquito`: **0.8903**

Le classi riconosciute meglio sono `Donut` (**0.9747**), `Baked Potato` (**0.9560**), `Chicken Curry` (**0.9500**) e `Crispy Chicken` (**0.9444**).

Rispetto all'esperimento 5 migliorano soprattutto `Crispy Chicken` (da 0.9259 a **0.9444**), `Hot Dog` (da 0.9231 a **0.9404**) e `Taco` (da 0.8606 a **0.8742**), mentre `Omelette` e `Chicken Curry` perdono meno di un punto.

Le principali confusioni sono:

- `Taquito → Taco`: **10 immagini**
- `Apple Pie → Cheesecake`: **9 immagini**
- `Ice Cream → Cheesecake`: **9 immagini**
- `Taco → Taquito` e `Cheesecake → Apple Pie`: **6 immagini** ciascuna
- `Apple Pie → Omelette`: **5 immagini**

Gli errori sono più distribuiti che negli esperimenti precedenti: dalla settima coppia in poi ogni confusione riguarda al massimo 4 immagini. Le due aree difficili restano comunque **Taco/Taquito**, ora confusi in entrambe le direzioni, e i dessert **Apple Pie/Cheesecake/Ice Cream**.

#### Confronto con l'esperimento precedente

Rispetto all'esperimento 5, manteniamo fine-tuning completo e Label Smoothing e modifichiamo solamente l'augmentation da `moderate` a `strong`.

| Metrica | Augmentation moderate | Augmentation strong | Differenza |
|---|---:|---:|---:|
| Loss | 0.3780 | 0.3745 | -0.0035 |
| Accuracy | 0.9192 | 0.9228 | +0.0036 |
| F1 Macro | 0.9194 | 0.9229 | +0.0035 |
| Top-3 Accuracy | 0.9774 | 0.9743 | -0.0031 |

L'augmentation `strong` produce un **piccolo miglioramento** di Accuracy e F1 Macro e una validation loss leggermente più bassa, mentre la Top-3 Accuracy diminuisce di poco. La differenza di F1 Macro è di soli **0.35 punti**, circa 8 immagini su 2.214: con un solo seed non basta per affermare che `strong` sia migliore di `moderate` in generale.

L'effetto più evidente è la **riduzione del divario tra training e validation**, da 6.6 a circa 5 punti di F1 Macro: l'augmentation più intensa agisce come regolarizzazione.

#### Conclusione

Il passaggio da augmentation `moderate` a `strong` porta a un miglioramento contenuto delle prestazioni di validation e a un divario più ridotto con il training. Il **F1 Macro di 0.9229** è il valore più alto ottenuto finora.

Nel prossimo esperimento manteniamo l'augmentation `strong` e il Label Smoothing e introduciamo **Mixup**, per verificare se questa ulteriore tecnica di regolarizzazione permette di migliorare ancora la generalizzazione.

In [ ]:
# ============================================================
# TEST 7 — Augmentation forte e Mixup
# ============================================================
# Confronto con il test 6: aggiungiamo Mixup con alpha 0.2. Il F1 di training non è riportato per le immagini miscelate; il confronto usa il F1 di validation.
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(mixup_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

mixup_metrics = evaluator.full_report(history=result["history"])
mixup_result = result
mixup_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
mixup_result["model"].to("cpu")
del runner

### Esperimento 7 — Augmentation strong + Mixup

Nel settimo esperimento manteniamo **EfficientNet-B0 completamente allenabile**, l'augmentation `strong` e il **Label Smoothing = 0.1**, introducendo **Mixup con alpha = 0.2**.

Mixup genera durante il training nuovi esempi combinando coppie di immagini e le rispettive etichette. Ad esempio, un'immagine utilizzata per il training potrebbe essere composta per il **70% da Taco e per il 30% da Taquito**, con un target che riflette la stessa combinazione.

L'obiettivo è verificare se questa ulteriore forma di regolarizzazione permette di migliorare la generalizzazione rispetto alla configurazione precedente.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 25 (ultima epoca disponibile) |
| Loss | 0.4537 |
| Accuracy | 0.9210 |
| F1 Macro | 0.9211 |
| F1 Weighted | 0.9212 |
| Top-3 Accuracy | 0.9734 |

#### Perché Accuracy e F1 Macro del training sono `NaN`?

Durante il training con Mixup le etichette non rappresentano più necessariamente una singola classe.

Senza Mixup possiamo avere, ad esempio:

`classe reale = Taco`

e confrontarla direttamente con:

`classe predetta = Taco`

Con Mixup, invece, il target può diventare ad esempio:

`70% Taco + 30% Taquito`

Non esiste quindi una singola classe reale con cui effettuare il normale confronto utilizzato per calcolare Accuracy e F1 Macro.

Per questo motivo, quando Mixup è attivo, la pipeline **non calcola intenzionalmente Accuracy e F1 Macro sul training** e restituisce `NaN` per queste due metriche. Non si tratta quindi di un errore o di un problema numerico. Per lo stesso motivo nel grafico del F1 Macro compare solo la curva di validation.

La **training loss viene invece calcolata normalmente**, perché la funzione di loss utilizzata con Mixup è in grado di lavorare con questi target misti.

Sul validation set Mixup non viene applicato: ogni immagine mantiene la propria etichetta originale e Accuracy, F1 Macro e le altre metriche vengono quindi calcolate normalmente.

#### Analisi dei risultati

Il modello raggiunge il risultato migliore all'**ultima epoca (25)**, con un **F1 Macro di 0.9211**, un'**Accuracy del 92.10%** e una **Top-3 Accuracy del 97.34%**.

Il F1 Macro di validation cresce più lentamente e con più oscillazioni rispetto all'esperimento 6: passa da **0.8271** alla prima epoca a **0.9082** all'undicesima e poi migliora a piccoli passi (**0.9132** all'epoca 15, **0.9196** alla 21, **0.9211** alla 25). L'early stopping non interviene e il training si ferma per il limite di 25 epoche: il modello converge più lentamente e potrebbe migliorare ancora di poco con più epoche, come già osservato nell'esperimento 2 con l'augmentation `moderate`.

La validation loss è più alta e più irregolare che nell'esperimento 6: oscilla tra **0.41 e 0.58** e alla migliore epoca vale **0.4537**, contro **0.3745**. Mixup e Label Smoothing insieme rendono le probabilità predette meno estreme anche quando la predizione è corretta, e questo aumenta la cross-entropy di validation pur con un'accuracy simile.

La training loss resta intorno a **1.0**, molto più alta che negli altri esperimenti: con Mixup il modello deve prevedere target misti, che non possono essere riprodotti con probabilità vicine a 1. Poiché Accuracy e F1 Macro non vengono calcolati sul training, non è possibile misurare il divario tra training e validation con queste metriche come negli esperimenti precedenti.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Taco`: **0.8544**
- `Taquito`: **0.8718**
- `Cheesecake`: **0.8855**
- `Sandwich`: **0.8868**

Le classi riconosciute meglio sono `Donut` (**0.9718**), `Sushi` (**0.9684**), `Fries` (**0.9565**) e `Baked Potato` (**0.9487**).

Rispetto all'esperimento 6, Mixup **sposta gli errori tra le classi più che ridurli**. Migliorano `Apple Pie` (da 0.8726 a **0.9010**), `Sushi` (da 0.9423 a **0.9684**) e `Fries` (da 0.9360 a **0.9565**); peggiorano invece `Sandwich` (da 0.9186 a **0.8868**), `Taquito` (da 0.8903 a **0.8718**) e `Taco` (da 0.8742 a **0.8544**).

Le principali confusioni sono:

- `Taquito → Taco`: **13 immagini**
- `Ice Cream → Cheesecake`: **12 immagini**
- `Apple Pie → Cheesecake`: **10 immagini**
- `Taco → Taquito`: **8 immagini**
- `Chicken Curry → Omelette`: **7 immagini**

La coppia `Taco` / `Taquito` resta la principale fonte di errore, confusa in entrambe le direzioni (21 immagini in totale, contro 16 nell'esperimento 6).

#### Confronto con l'esperimento precedente

Rispetto all'esperimento 6, manteniamo augmentation `strong` e Label Smoothing e introduciamo Mixup.

| Metrica | Strong + Label Smoothing | Strong + Mixup | Differenza |
|---|---:|---:|---:|
| Loss | 0.3745 | 0.4537 | +0.0792 |
| Accuracy | 0.9228 | 0.9210 | -0.0018 |
| F1 Macro | 0.9229 | 0.9211 | -0.0018 |
| Top-3 Accuracy | 0.9743 | 0.9734 | -0.0009 |

L'introduzione di Mixup **non migliora le prestazioni sul validation set**. Il F1 Macro scende di soli **0.18 punti**, circa 4 immagini su 2.214: le due configurazioni sono di fatto equivalenti per correttezza delle predizioni. Mixup produce però una validation loss nettamente più alta e una convergenza più lenta.

#### Conclusione

Mixup permette di ottenere prestazioni elevate, ma in questo esperimento **non porta un vantaggio rispetto alla configurazione con augmentation strong e Label Smoothing**: il F1 Macro è praticamente identico (**0.9211** contro **0.9229**), la validation loss è più alta e il modello impiega più epoche per convergere.

Tra i sette esperimenti controllati, la configurazione con il miglior F1 Macro di validation rimane quindi quella dell'**esperimento 6**. Nell'esperimento di approfondimento successivo verifichiamo se, partendo da questa configurazione, una risoluzione di input maggiore aiuta a distinguere meglio le classi visivamente simili.

## 8. Esperimento di approfondimento: risoluzione più alta

Gli errori residui dei sette esperimenti controllati riguardano soprattutto **dettagli fini** tra piatti simili (Taco/Taquito, Apple Pie/Cheesecake/Ice Cream). Una risoluzione di input maggiore conserva più dettagli: l'esperimento 8 ripete la configurazione dell'esperimento 6 cambiando solo la dimensione delle immagini.

| # | Nome | Confronto | Unico fattore modificato |
|---|---|---|---|
| 8 | strong_aug_label_smoothing_288px | 8 vs 6 | risoluzione di input da 224 a 288 pixel |

Viene mantenuto EfficientNet-B0: a 288 pixel il costo di calcolo cresce di circa 1.65 volte ed è modificato un solo fattore, perché l'architettura resta la stessa. L'effetto di un modello più grande viene verificato separatamente nell'esperimento 9.

In [ ]:
# ============================================================
# TEST 8 — Esperimento 6 con immagini 288x288
# ============================================================
# Confronto con il test 6: cambia soltanto la risoluzione di input, da 224 a 288 pixel.
# Ogni epoca richiede più calcolo (circa 1.65 volte): con Colab gratuito conviene
# lasciare attivo il salvataggio dei checkpoint su Drive (cella 1B).
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(highres_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

highres_metrics = evaluator.full_report(history=result["history"])
highres_result = result
highres_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
highres_result["model"].to("cpu")
del runner

### Esperimento 8 — Augmentation strong + Label Smoothing con risoluzione 288x288

Nell'esperimento di approfondimento manteniamo esattamente la configurazione dell'esperimento 6: **EfficientNet-B0 completamente allenabile**, augmentation `strong` e **Label Smoothing = 0.1**.

L'unica modifica riguarda la **risoluzione delle immagini in input**, che passa da 224x224 a **288x288 pixel**.

L'obiettivo è verificare se conservare più dettagli riduce le confusioni tra classi visivamente simili, in particolare Taco/Taquito e Apple Pie/Cheesecake/Ice Cream.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 13 (early stopping all'epoca 17) |
| Loss | 0.3661 |
| Accuracy | 0.9282 |
| F1 Macro | 0.9281 |
| F1 Weighted | 0.9282 |
| Top-3 Accuracy | 0.9783 |

#### Analisi dei risultati

Con immagini a 288x288 il modello raggiunge il risultato migliore all'**epoca 13**, con un **F1 Macro di 0.9281**, un'**Accuracy del 92.82%** e una **Top-3 Accuracy del 97.83%**. Dopo quattro epoche senza miglioramenti interviene l'**early stopping** all'epoca 17.

Il F1 Macro di validation parte da **0.8582** e supera **0.90** già all'epoca 4. Dopo tre epoche senza miglioramenti lo scheduler riduce il learning rate e all'epoca 8 il F1 sale a **0.9196**, per poi arrivare a **0.9243** all'epoca 10 e al massimo di **0.9281** all'epoca 13. Il modello converge **più rapidamente** rispetto all'esperimento 6, che aveva raggiunto il suo miglior valore all'epoca 20.

Anche la validation loss è la più bassa tra tutti gli esperimenti: scende a **0.3510** all'epoca 11 e resta tra **0.35 e 0.39** nelle epoche successive. Come negli esperimenti 5 e 6, la training loss resta intorno a **0.65** per effetto del Label Smoothing.

Il divario tra training e validation è il più contenuto tra gli esperimenti con fine-tuning: alla migliore epoca il F1 Macro è **0.9643 sul training** e **0.9281 sulla validation**, circa **3.6 punti**, contro i circa 5 dell'esperimento 6.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Taco`: **0.8726**
- `Taquito`: **0.8787**
- `Cheesecake`: **0.8982**
- `Apple Pie`: **0.9103**

Le classi riconosciute meglio sono `Donut` (**0.9779**), `Sushi` (**0.9587**), `Chicken Curry` (**0.9536**) e `Fries` (**0.9428**).

Rispetto all'esperimento 6 la risoluzione più alta migliora soprattutto i **dessert** e i piatti con ingredienti misti: `Apple Pie` passa da 0.8726 a **0.9103**, `Ice Cream` da 0.9206 a **0.9385**, `Cheesecake` da 0.8862 a **0.8982** e `Omelette` da 0.9136 a **0.9329**. Per questi piatti i dettagli di consistenza e superficie, meglio conservati a 288 pixel, aiutano la distinzione.

La coppia **Taco/Taquito** non migliora: `Taco` resta a **0.8726** e `Taquito` scende da 0.8903 a **0.8787**. Le confusioni tra le due classi sono **21** in totale (12 + 9), contro le 16 dell'esperimento 6. Questa difficoltà non sembra dipendere dalla quantità di dettagli nell'immagine: i due piatti hanno ingredienti simili e si distinguono soprattutto per la forma (tortilla piegata o arrotolata), che può essere ambigua o poco visibile nella foto. Calano leggermente anche `Baked Potato` (da 0.9560 a **0.9390**) e `Crispy Chicken` (da 0.9444 a **0.9346**).

Le principali confusioni sono:

- `Taquito → Taco`: **12 immagini**
- `Ice Cream → Cheesecake`: **10 immagini**
- `Taco → Taquito`: **9 immagini**
- `Apple Pie → Cheesecake`: **7 immagini**
- `Sandwich → Baked Potato`: **6 immagini**
- `Apple Pie → Omelette`: **5 immagini**

#### Confronto con l'esperimento 6

Rispetto all'esperimento 6 cambia soltanto la risoluzione di input, da 224 a 288 pixel.

| Metrica | 224x224 | 288x288 | Differenza |
|---|---:|---:|---:|
| Loss | 0.3745 | 0.3661 | -0.0084 |
| Accuracy | 0.9228 | 0.9282 | +0.0054 |
| F1 Macro | 0.9229 | 0.9281 | +0.0052 |
| Top-3 Accuracy | 0.9743 | 0.9783 | +0.0040 |

La risoluzione più alta migliora **tutte le metriche**. La differenza di F1 Macro è di **0.52 punti**, circa 12 immagini su 2.214: resta sotto il punto percentuale e, con un solo seed, non basta da sola a dimostrare un vantaggio stabile. È però coerente con gli altri segnali osservati: validation loss più bassa, divario più contenuto con il training, convergenza più rapida e un miglioramento concentrato proprio sulle classi di dessert che l'esperimento voleva aiutare.

#### Conclusione

L'aumento della risoluzione a 288x288 porta al **miglior F1 Macro di validation tra tutti gli esperimenti (0.9281)**, con la validation loss più bassa e senza aumentare il tempo complessivo di training.

Il miglioramento riguarda soprattutto i dessert (`Apple Pie`, `Cheesecake`, `Ice Cream`) e `Omelette`, mentre la confusione tra **Taco e Taquito** rimane il principale limite del modello e non viene risolta da una risoluzione maggiore.

Nell'esperimento 9 viene verificato se un modello più grande, EfficientNet-B2, riesce a ridurre anche questa confusione.

## 9. Esperimento di approfondimento: modello più grande

Nell'esperimento 8 la risoluzione maggiore ha ridotto le confusioni tra dessert, ma non quella tra **Taco e Taquito**. L'esperimento 9 verifica se un modello più capace distingue meglio questi piatti: ripete la configurazione dell'esperimento 8 sostituendo EfficientNet-B0 con **EfficientNet-B2**.

| # | Nome | Confronto | Unico fattore modificato |
|---|---|---|---|
| 9 | strong_aug_label_smoothing_288px_b2 | 9 vs 8 | architettura da EfficientNet-B0 a EfficientNet-B2 |

EfficientNet-B2 appartiene alla stessa famiglia di B0, ma è più largo e più profondo: con 14 classi ha circa 7.7 milioni di parametri contro 4.0. Restano invariati risoluzione (288x288), augmentation `strong`, Label Smoothing, fine-tuning completo e iperparametri di training; mean e std sono quelle del pretraining su ImageNet, uguali per i due modelli.

In [ ]:
# ============================================================
# TEST 9 — Esperimento 8 con EfficientNet-B2
# ============================================================
# Confronto con il test 8: cambia soltanto l'architettura, da EfficientNet-B0 a EfficientNet-B2.
# È l'esperimento più pesante (circa il doppio del calcolo del test 8): con Colab gratuito
# conviene lasciare attivo il salvataggio dei checkpoint su Drive (cella 1B).
# Valutiamo sul validation set; il test set resta riservato alla fase finale.
# ============================================================

config = copy.deepcopy(b2_cfg)

runner = ExperimentRunner(config, device=device)
result = runner.run()

evaluator = Evaluator(
    model=result["model"],
    loader=result["val_loader"],
    criterion=result["criterion"],
    device=device,
    class_names=result["class_names"],
    split_name="validation",
    use_amp=config.use_amp,
)

b2_metrics = evaluator.full_report(history=result["history"])
b2_result = result
b2_run_cfg = config  # conserva la configurazione effettivamente eseguita

# Conserviamo il modello in RAM CPU per liberare memoria GPU prima del test successivo.
b2_result["model"].to("cpu")
del runner

### Esperimento 9 — Esperimento 8 con EfficientNet-B2

Nel secondo esperimento di approfondimento viene mantenuta esattamente la configurazione dell'esperimento 8: immagini **288x288**, **fine-tuning completo**, augmentation `strong` e **Label Smoothing = 0.1**.

L'unica modifica riguarda l'architettura, che passa da **EfficientNet-B0** a **EfficientNet-B2**, un modello della stessa famiglia ma più grande (circa 7.7 milioni di parametri contro 4.0).

L'obiettivo è verificare se un modello più capace riduce le confusioni che la sola risoluzione non ha risolto, in particolare tra Taco e Taquito.

| Metrica (validation) | Valore |
|---|---:|
| Migliore epoca | 11 (early stopping all'epoca 15) |
| Loss | 0.3553 |
| Accuracy | 0.9255 |
| F1 Macro | 0.9257 |
| F1 Weighted | 0.9257 |
| Top-3 Accuracy | 0.9797 |

#### Analisi dei risultati

Con EfficientNet-B2 il modello raggiunge il risultato migliore all'**epoca 11**, con un **F1 Macro di 0.9257**, un'**Accuracy del 92.55%** e una **Top-3 Accuracy del 97.97%**, la più alta tra tutti gli esperimenti. Dopo quattro epoche senza miglioramenti interviene l'**early stopping** all'epoca 15.

Il modello parte molto bene: già alla prima epoca il F1 Macro di validation è **0.8786**, il valore iniziale più alto di tutti gli esperimenti, e alla seconda arriva a **0.8988**. Segue una fase di stallo tra **0.896 e 0.908** fino all'epoca 9; dopo la riduzione del learning rate da parte dello scheduler il F1 sale a **0.9149** all'epoca 10 e a **0.9257** all'epoca 11, per poi oscillare senza superare questo valore.

La validation loss resta bassa per tutto il training (tra **0.35 e 0.49**) e raggiunge **0.3526** all'epoca 14, il valore più basso tra tutti gli esperimenti, di poco inferiore a quello dell'esperimento 8.

Alla migliore epoca il F1 Macro è **0.9679 sul training** e **0.9257 sulla validation**: il divario (circa **4.2 punti**) è leggermente più ampio di quello dell'esperimento 8 (circa 3.6), coerente con la maggiore capacità del modello.

#### Analisi delle classi

Gli F1-score più bassi si osservano per:

- `Taco`: **0.8652**
- `Apple Pie`: **0.8834**
- `Cheesecake`: **0.8957**
- `Taquito`: **0.9068**

Le classi riconosciute meglio sono `Donut` (**0.9810**), `Baked Potato` (**0.9747**), `Fries` (**0.9627**) e `Sushi` (**0.9464**).

Rispetto all'esperimento 8, EfficientNet-B2 **sposta gli errori tra le classi più che ridurli**:

- migliorano `Taquito` (da 0.8787 a **0.9068**, con il recall che sale da 0.8535 a **0.8981**), `Baked Potato` (da 0.9390 a **0.9747**) e `Fries` (da 0.9428 a **0.9627**);
- peggiorano invece `Apple Pie` (da 0.9103 a **0.8834**), `Ice Cream` (da 0.9385 a **0.9150**), `Omelette` (da 0.9329 a **0.9103**) e `Chicken Curry` (da 0.9536 a **0.9354**).

Le confusioni tra **Taco e Taquito** diminuiscono: `Taquito → Taco` scende da 12 a **10 immagini** e `Taco → Taquito` esce dalle dieci coppie più confuse. `Taco` non migliora però nel complesso (da 0.8726 a **0.8652**), perché compare una nuova confusione con `Hot Dog`. Nei dessert si perde invece parte del miglioramento ottenuto con la risoluzione più alta.

Le principali confusioni sono:

- `Taquito → Taco`: **10 immagini**
- `Taco → Hot Dog` e `Ice Cream → Apple Pie`: **8 immagini** ciascuna
- `Apple Pie → Cheesecake`, `Cheesecake → Apple Pie`, `Ice Cream → Cheesecake` e `Omelette → Chicken Curry`: **7 immagini** ciascuna
- `Omelette → Apple Pie`: **6 immagini**

#### Confronto con l'esperimento 8

Rispetto all'esperimento 8 cambia soltanto l'architettura, da EfficientNet-B0 a EfficientNet-B2.

| Metrica | EfficientNet-B0 | EfficientNet-B2 | Differenza |
|---|---:|---:|---:|
| Loss | 0.3661 | 0.3553 | -0.0108 |
| Accuracy | 0.9282 | 0.9255 | -0.0027 |
| F1 Macro | 0.9281 | 0.9257 | -0.0024 |
| Top-3 Accuracy | 0.9783 | 0.9797 | +0.0014 |

EfficientNet-B2 ottiene una validation loss e una Top-3 Accuracy leggermente migliori, ma Accuracy e F1 Macro leggermente inferiori. La differenza di F1 Macro è di soli **0.24 punti**, circa 5 immagini su 2.214: i due modelli sono di fatto **equivalenti**.

#### Conclusione

Un modello quasi doppio per numero di parametri **non porta un miglioramento complessivo**: il F1 Macro (**0.9257**) resta allo stesso livello dell'esperimento 8 (**0.9281**). EfficientNet-B2 riduce le confusioni tra Taco e Taquito, ma perde qualcosa sui dessert, e gli errori si spostano tra le classi senza diminuire.

Questo risultato suggerisce che, su questo dataset, il limite principale non è la capacità del modello: le difficoltà residue dipendono soprattutto dalla somiglianza tra alcuni piatti e dalla qualità di alcune immagini ed etichette. A parità di prestazioni, **EfficientNet-B0 a 288 pixel resta preferibile** perché più leggero, sia in training sia in utilizzo.

Nella sezione successiva il confronto automatico seleziona, esclusivamente sulla base del F1 Macro di validation, la configurazione da valutare sul test set.

## 10. Confronto finale e selezione del modello

I risultati ottenuti dai diversi esperimenti vengono confrontati utilizzando **esclusivamente il validation set**, senza effettuare nuovi addestramenti.

Come criterio di selezione viene utilizzato il **F1 Macro di validation**, perché assegna lo stesso peso a ciascuna delle 14 classi e permette quindi di valutare le prestazioni complessive senza privilegiare le classi più rappresentate.

Il modello selezionato è quello con il **F1 Macro di validation più alto**, considerando il valore a precisione completa. I valori arrotondati vengono utilizzati solamente per la visualizzazione dei risultati.

In caso di perfetta parità del F1 Macro, viene mantenuto il primo esperimento secondo l'ordine predefinito.

In [ ]:
# ============================================================
# CELLA 12 — Confronto tra gli esperimenti
# ============================================================
# REGOLA: solo il validation set viene usato per confrontare gli esperimenti.
# Il test set verrà usato UNA SOLA VOLTA alla fine, sul modello scelto.
# ============================================================

# Raccogliamo i risultati già calcolati nelle celle TEST, senza riaddestrare.
experiment_records = [
    (baseline_run_cfg, baseline_metrics),
    (frozen_run_cfg, frozen_metrics),
    (progressive_run_cfg, progressive_metrics),
    (full_ft_run_cfg, full_ft_metrics),
    (smoothing_run_cfg, smoothing_metrics),
    (strong_aug_run_cfg, strong_aug_metrics),
    (mixup_run_cfg, mixup_metrics),
    (highres_run_cfg, highres_metrics),
    (b2_run_cfg, b2_metrics),
]

result_lookup = {
    baseline_run_cfg.experiment_name: baseline_result,
    frozen_run_cfg.experiment_name: frozen_result,
    progressive_run_cfg.experiment_name: progressive_result,
    full_ft_run_cfg.experiment_name: full_ft_result,
    smoothing_run_cfg.experiment_name: smoothing_result,
    strong_aug_run_cfg.experiment_name: strong_aug_result,
    mixup_run_cfg.experiment_name: mixup_result,
    highres_run_cfg.experiment_name: highres_result,
    b2_run_cfg.experiment_name: b2_result,
}

def finetuning_label(cfg):
    if cfg.freeze_backbone and not cfg.progressive_unfreeze:
        return "solo testa"
    if cfg.progressive_unfreeze:
        return "due fasi"
    return "completo"

experiment_results = []
for cfg, metrics in experiment_records:
    experiment_results.append({
        "Esperimento": cfg.experiment_name,
        "Augmentation": cfg.augmentation,
        "Fine-tuning": finetuning_label(cfg),
        "Label smoothing": cfg.label_smoothing,
        "Mixup alpha": cfg.mixup_alpha,
        "Modello": cfg.model_name.replace("efficientnet_", "EfficientNet-").upper().replace("EFFICIENTNET", "EfficientNet"),
        "Risoluzione": cfg.img_size or 224,
        "Val Loss": metrics["loss"],
        "Val Accuracy": metrics["accuracy"],
        "Val F1 Macro": metrics["f1_macro"],
        "Val Top-3 Acc": metrics["top3_accuracy"],
    })

summary_df = pd.DataFrame(experiment_results).sort_values("Val F1 Macro", ascending=False, kind="stable").reset_index(drop=True)

print("RIEPILOGO CONFRONTO ESPERIMENTI (ordinato per Val F1 Macro decrescente):")
print(summary_df.round(4).to_string(index=False))

best_row = summary_df.iloc[0]
print(f"\nMIGLIOR ESPERIMENTO: {best_row['Esperimento']} — Val F1 Macro = {best_row['Val F1 Macro']:.4f}")

summary_df.round(4)


### Confronto finale degli esperimenti

I nove esperimenti (sette controllati e due di approfondimento) permettono di valutare progressivamente l'effetto delle diverse scelte di training, mantenendo il confronto il più possibile controllato. Tutti hanno lo stesso budget massimo di 25 epoche, con early stopping sul F1 Macro di validation.

| # | Esperimento | Modifica rispetto al precedente | Val F1 Macro | Differenza |
|---|---|---|---:|---:|
| 1 | baseline_basic | riferimento | 0.8547 | — |
| 2 | frozen_moderate | augmentation basic → moderate | 0.8654 | +1.07 |
| 3 | progressive_moderate | sblocco del backbone dall'epoca 4 (vs 2) | 0.9092 | +4.38 |
| 4 | full_ft_moderate | backbone allenabile dall'inizio (vs 2) | 0.9128 | +4.74 |
| 5 | full_ft_smoothing | Label Smoothing 0.1 (vs 4) | 0.9194 | +0.66 |
| 6 | strong_aug_label_smoothing | augmentation moderate → strong (vs 5) | 0.9229 | +0.35 |
| 7 | strong_aug_mixup | Mixup alpha 0.2 (vs 6) | 0.9211 | -0.18 |
| 8 | strong_aug_label_smoothing_288px | risoluzione 224 → 288 pixel (vs 6) | **0.9281** | +0.52 |
| 9 | strong_aug_label_smoothing_288px_b2 | EfficientNet-B0 → EfficientNet-B2 (vs 8) | 0.9257 | -0.24 |

Le differenze sono espresse in punti percentuali di F1 Macro.

Il primo risultato evidente riguarda il **fine-tuning del backbone**. Con il backbone congelato, la baseline raggiunge un F1 Macro di **0.8547** e l'augmentation `moderate` la porta a **0.8654** (+1.07 punti). Sbloccare il backbone produce invece il salto più grande dell'intero percorso: **0.9092** con il fine-tuning in due fasi e **0.9128** con il fine-tuning completo fin dall'inizio, circa **+4.5 punti** rispetto al backbone congelato. Tra le due strategie di fine-tuning la differenza è di soli 0.36 punti: in questo run il fine-tuning completo è leggermente superiore, ma la differenza è troppo piccola per concludere che sia in generale la strategia migliore.

Le tecniche di regolarizzazione aggiunte al fine-tuning completo producono miglioramenti più piccoli ma nella stessa direzione. Il **Label Smoothing** porta il F1 Macro a **0.9194** (+0.66) e rende la validation loss più bassa e stabile; l'augmentation **`strong`** lo porta a **0.9229** (+0.35) e riduce il divario tra training e validation. **Mixup** non aggiunge invece alcun vantaggio: il F1 Macro resta praticamente invariato (**0.9211**, -0.18), mentre la validation loss aumenta e la convergenza rallenta.

Il primo esperimento di approfondimento, con **immagini 288x288**, raggiunge il **miglior F1 Macro di validation (0.9281)**. Il miglioramento riguarda soprattutto i dessert (`Apple Pie`, `Cheesecake`, `Ice Cream`) e `Omelette`.

Il secondo, con **EfficientNet-B2** al posto di B0, non migliora ulteriormente: il F1 Macro è **0.9257** (-0.24 punti, circa 5 immagini), di fatto equivalente, anche se la validation loss (**0.3553**) e la Top-3 Accuracy (**0.9797**) sono leggermente migliori. Il modello più grande riduce le confusioni tra Taco e Taquito, ma perde qualcosa sui dessert: gli errori si spostano tra le classi senza diminuire. Su questo dataset il limite principale non sembra quindi la capacità del modello.

Queste differenze vanno lette tenendo conto della dimensione del validation set: con circa 2.200 immagini, una differenza di 0.5 punti corrisponde a una decina di immagini classificate diversamente. Con un solo seed, solo il **fine-tuning del backbone** (+4.4/4.7 punti, circa 100 immagini) mostra un miglioramento abbastanza ampio da essere considerato solido. Label Smoothing, augmentation `strong`, risoluzione maggiore e modello più grande danno ciascuno meno di un punto, e presi singolarmente potrebbero cambiare ripetendo l'addestramento con un altro seed. Nel loro insieme, però, vanno nella stessa direzione: dal fine-tuning completo senza regolarizzazione (**0.9128**) alla configurazione finale (**0.9281**) il guadagno complessivo è di **1.5 punti**, circa 34 immagini, accompagnato da validation loss più bassa e divario più ridotto con il training.

Due difficoltà restano presenti in tutti gli esperimenti: la confusione tra **Taco e Taquito**, che nessuna modifica ha ridotto in modo stabile (EfficientNet-B2 la riduce un po', ma a scapito dei dessert), e quella tra i dessert **Apple Pie, Cheesecake e Ice Cream**, che diminuisce con il fine-tuning e con la risoluzione più alta.

La configurazione selezionata è quindi **`strong_aug_label_smoothing_288px`** (esperimento 8), che ottiene il F1 Macro di validation più alto e, a parità di prestazioni con l'esperimento 9, usa anche il modello più leggero. Utilizza:

- EfficientNet-B0 pre-addestrata su ImageNet;
- fine-tuning completo del backbone;
- augmentation `strong`;
- Label Smoothing = **0.1**;
- nessun Mixup;
- immagini in input **288x288**.

Sul validation set questa configurazione raggiunge:

| Metrica | Valore |
|---|---:|
| Loss | **0.3661** |
| Accuracy | **0.9282** |
| F1 Macro | **0.9281** |
| F1 Weighted | **0.9282** |
| Top-3 Accuracy | **0.9783** |

Rispetto alla baseline il F1 Macro migliora complessivamente di **7.3 punti percentuali** (da 0.8547 a 0.9281).

La scelta del modello finale viene effettuata **esclusivamente sulla validation**. Il test set non viene utilizzato per confrontare o selezionare gli esperimenti e viene usato solo per la valutazione finale del modello selezionato.

## 11. Valutazione finale sul test set

Dopo aver confrontato gli esperimenti sul **validation set**, viene selezionata automaticamente la configurazione con il **F1 Macro di validation più alto**.

Soltanto a questo punto il modello selezionato viene valutato sul **test set**, che non ha partecipato al confronto tra gli esperimenti né alla scelta degli iperparametri.

La valutazione finale comprende **loss, accuracy, F1 Macro, F1 Weighted e Top-3 Accuracy**. Vengono inoltre generati il **classification report**, la **matrice di confusione normalizzata** e l'analisi delle **coppie di classi più frequentemente confuse**, in modo da osservare le prestazioni sia complessive sia sulle singole classi.

In [ ]:
# ============================================================
# CELLA 13 — Valutazione finale sul test set
# ============================================================
# Seleziona automaticamente il risultato dell'esperimento con il Val F1 Macro
# più alto (calcolato nella cella di confronto) e lo valuta sul test set.
# ============================================================

best_experiment_name = best_row["Esperimento"]
cfg_lookup = {cfg.experiment_name: cfg for cfg, _ in experiment_records}

best_result = result_lookup[best_experiment_name]
best_cfg = cfg_lookup[best_experiment_name]
best_result["model"].to(device)

print(f"Modello scelto per la valutazione finale: {best_experiment_name}")
print(f"(Val F1 Macro = {best_row['Val F1 Macro']:.4f})")

test_evaluator = Evaluator(
    model=best_result["model"],
    loader=best_result["test_loader"],
    criterion=best_result["criterion"],
    device=device,
    class_names=best_result["class_names"],
    split_name="test",
    use_amp=best_cfg.use_amp,
)

test_metrics_final = test_evaluator.full_report(history=best_result["history"])

### Risultati sul test set

Dopo aver confrontato i nove esperimenti esclusivamente sul validation set, è stato selezionato il modello `strong_aug_label_smoothing_288px`, che aveva ottenuto il miglior **F1 Macro di validation pari a 0.9281**.

Il modello selezionato viene quindi valutato sul **test set** (2.792 immagini, da 197 a 200 per classe), che non è stato utilizzato né per l'addestramento né per la scelta della configurazione migliore.

#### Risultati finali

| Metrica | Validation | Test | Differenza |
|---|---:|---:|---:|
| Loss | 0.3661 | 0.3917 | +0.0256 |
| Accuracy | 0.9282 | 0.9108 | -0.0174 |
| F1 Macro | 0.9281 | 0.9106 | -0.0175 |
| F1 Weighted | 0.9282 | 0.9105 | -0.0177 |
| Top-3 Accuracy | 0.9783 | 0.9735 | -0.0048 |

#### Confronto tra validation e test

Sul test set il modello raggiunge un'**Accuracy del 91.08%** e un **F1 Macro di 0.9106**.

Rispetto alla validation si osserva una riduzione contenuta delle prestazioni: il F1 Macro passa da **0.9281 a 0.9106** e l'Accuracy da **0.9282 a 0.9108**, un calo di circa **1.75 punti percentuali** per entrambe le metriche. La **Top-3 Accuracy** diminuisce di meno di mezzo punto (da **0.9783 a 0.9735**): nel **97.35%** delle immagini di test la classe corretta compare tra le tre classi considerate più probabili dal modello.

Un calo di questo tipo è atteso. La configurazione e l'epoca del checkpoint sono state scelte proprio perché ottenevano il miglior risultato sulla validation, e questa scelta tende a rendere la stima di validation leggermente ottimistica; contribuiscono inoltre la variabilità dovuta al campione di immagini e le possibili differenze tra i due split. Con un solo seed e un solo test set non è possibile attribuire il calo a una causa precisa.

La vicinanza tra i risultati di validation e test indica comunque che le prestazioni osservate durante la selezione del modello si mantengono bene anche su immagini non usate per le decisioni.

#### Analisi delle singole classi

Le prestazioni non sono completamente uniformi tra le 14 classi.

Tra le classi con F1-score più elevato troviamo:

- `Sushi`: **0.9548**
- `Donut`: **0.9531**
- `Fries`: **0.9364**
- `Crispy Chicken`: **0.9296**

Le classi più difficili risultano invece:

- `Taco`: **0.8217**
- `Cheesecake`: **0.8822**
- `Apple Pie`: **0.8861**
- `Taquito`: **0.8934**

`Taco` è la classe con il calo maggiore rispetto alla validation (da 0.8726 a **0.8217**) e ha un **recall di 0.7950**: circa un'immagine su cinque realmente appartenente a questa classe viene attribuita ad altre classi, non solo a `Taquito` ma anche a `Crispy Chicken` e `Baked Potato`.

`Baked Potato` mostra invece il comportamento opposto: **recall di 0.9849** ma **precision di 0.8711**. Il modello riconosce quasi tutte le patate al forno, ma assegna a questa classe anche immagini di altri piatti, in particolare `Taco`, `Crispy Chicken`, `Fries` e `Taquito`.

Anche i dessert perdono qualche punto rispetto alla validation, in particolare `Ice Cream` (da 0.9385 a **0.8990**).

#### Analisi degli errori

La matrice di confusione conferma le difficoltà già osservate durante la validation.

Le principali confusioni sul test set sono:

- `Ice Cream → Cheesecake`: **13 immagini**
- `Taco → Taquito`: **12 immagini**
- `Taco → Crispy Chicken`: **10 immagini**
- `Taquito → Taco`: **10 immagini**
- `Hot Dog → Taco`, `Taco → Baked Potato`, `Apple Pie → Ice Cream`, `Apple Pie → Omelette`, `Cheesecake → Apple Pie` e `Chicken Curry → Omelette`: **8 immagini** ciascuna

Gli errori si concentrano in due aree. La prima riguarda `Taco`, coinvolto in cinque delle dieci coppie più confuse: oltre alla confusione reciproca con `Taquito` (22 immagini in totale), viene scambiato con altri piatti a base di carne e tortilla o pane (`Crispy Chicken`, `Baked Potato`, `Hot Dog`). La seconda riguarda i dessert `Apple Pie`, `Cheesecake` e `Ice Cream`, a cui si aggiunge `Omelette`, confusa con `Apple Pie` e `Chicken Curry`.

#### Conclusione finale

Il modello `strong_aug_label_smoothing_288px`, selezionato sulla base del validation set, raggiunge sul test set un **F1 Macro di 0.9106**, un'**Accuracy del 91.08%** e una **Top-3 Accuracy del 97.35%**.

Il calo rispetto alla validation è limitato a circa **1.75 punti percentuali**. I risultati mostrano quindi che il modello mantiene prestazioni elevate anche sul test set, pur evidenziando difficoltà specifiche nella classe `Taco` e nella distinzione tra alcuni dessert.

L'intero confronto tra gli esperimenti è stato effettuato utilizzando la validation, mentre il test set è stato utilizzato solo dopo la scelta della configurazione finale. La valutazione sul test fornisce quindi una stima delle prestazioni su dati non usati né per l'addestramento né per la selezione del modello, con il limite descritto nelle sezioni 7 e 8: il test era già stato consultato in prove precedenti e nel primo run con budget di 15 epoche (F1 Macro di test 0.9015 per il modello allora selezionato).

## 12. Analisi qualitativa degli errori

Le metriche aggregate permettono di misurare **quanto spesso il modello commette errori**, ma non permettono di comprenderne direttamente le possibili cause.

Per questo motivo viene effettuata anche un'**analisi qualitativa** di alcune immagini del test set classificate in modo errato dal modello finale. Per ogni esempio vengono mostrate l'immagine, la **classe reale** e la **classe predetta**.

L'osservazione diretta degli errori permette di verificare se alcune classificazioni errate possono essere associate a **somiglianze visive tra classi**, immagini ambigue o poco rappresentative, oppure a possibili **anomalie nelle etichette del dataset**.

In [ ]:
# ============================================================
# CELLA 14 — Esempi di predizioni sbagliate sul test set
# ============================================================

def collect_wrong_predictions(loader, metrics, max_examples=12):
    # Riusa le predizioni del report finale, senza una seconda inferenza sul test.
    wrong = []
    for index, (true_label, pred_label) in enumerate(zip(metrics["y_true"], metrics["y_pred"])):
        if true_label != pred_label:
            image, _ = loader.dataset[index]
            wrong.append((image, int(true_label), int(pred_label)))
        if len(wrong) >= max_examples:
            break
    return wrong

# Le immagini nel test_loader sono già normalizzate (mean/std di ImageNet):
# per mostrarle serve invertire la normalizzazione, altrimenti i colori risultano innaturali.
mean = torch.tensor(best_result["data_cfg"]["mean"]).view(3, 1, 1)
std = torch.tensor(best_result["data_cfg"]["std"]).view(3, 1, 1)

def denormalize(img_tensor):
    img = img_tensor * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()

wrong_examples = collect_wrong_predictions(best_result["test_loader"], test_metrics_final)

if len(wrong_examples) > 0:
    n_cols = 4
    n_rows = (len(wrong_examples) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    for ax, (img, true_idx, pred_idx) in zip(np.array(axes).flat, wrong_examples):
        ax.imshow(denormalize(img))
        true_name = DISPLAY_NAME.get(best_result["class_names"][true_idx], best_result["class_names"][true_idx])
        pred_name = DISPLAY_NAME.get(best_result["class_names"][pred_idx], best_result["class_names"][pred_idx])
        ax.set_title(f"Vero: {true_name}\nPredetto: {pred_name}", fontsize=9, color="#c44e52")
        ax.axis("off")
    for ax in np.array(axes).flat[len(wrong_examples):]:
        ax.axis("off")
    plt.suptitle("Esempi di classificazioni errate sul test set", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Nessuna predizione errata da mostrare.")


### Analisi qualitativa delle classificazioni errate

Per completare l'analisi quantitativa sono state osservate alcune immagini del **test set classificate in modo errato dal modello finale**. Per ogni esempio vengono mostrate l'etichetta reale presente nel dataset e la classe predetta dal modello.

La griglia mostra i primi 12 errori nell'ordine del test set, quindi riguarda solo le prime classi in ordine alfabetico: 3 immagini di `Baked Potato` e 9 di `Crispy Chicken`. Non è un campione rappresentativo di tutti gli errori (le confusioni più frequenti, come Taco/Taquito e i dessert, non compaiono), ma permette di osservare da vicino alcuni tipi di errore ricorrenti.

Gli errori osservati si possono raggruppare in tre categorie.

**1. Predizioni plausibili per piatti che appartengono a più categorie.** Tre immagini etichettate come `Crispy Chicken` mostrano in realtà un **hamburger di pollo fritto**, cioè un panino con una cotoletta impanata. Il modello le classifica come `Sandwich`: la predizione non è sbagliata nel contenuto, perché il piatto è sia un pollo fritto sia un panino. Il dataset assegna però a ogni immagine una sola etichetta, e in questi casi le due classi si sovrappongono. Questi errori dipendono più dalla definizione delle categorie che da un limite del modello.

**2. Immagini poco rappresentative o con etichetta discutibile.** Alcune immagini non mostrano il piatto previsto dalla classe in modo chiaro:

- due immagini sono **confezioni di prodotti** e non piatti: una lattina decorata con disegni di patate al forno (etichetta `Baked Potato`, predetta `Hot Dog`) e la confezione di un burger di pollo impanato (etichetta `Crispy Chicken`, predetta `Donut`);
- un'immagine etichettata `Baked Potato` mostra **patatine in busta con una salsa**, senza alcuna patata al forno;
- un'immagine etichettata `Baked Potato` mostra **patate a spicchi** con la scritta "potato wedges", un piatto diverso dalla patata al forno intera;
- due immagini etichettate `Crispy Chicken` mostrano pollo **non fritto**: cosce di pollo arrosto glassate e un pollo cotto in casseruola.

In questi casi l'errore dipende almeno in parte dalla **qualità delle etichette** e dalla presenza nel dataset di immagini che non rappresentano un piatto reale, come confezioni o immagini con testo sovrapposto.

**3. Somiglianze visive di forma e condimento.** Gli altri errori seguono somiglianze visive riconoscibili:

- forme **allungate, dorate e croccanti** portano alla classe `Taquito`: gli spicchi di patata e i filetti di pollo impanato vengono scambiati per taquitos;
- piatti coperti da una **salsa bianca e cremosa** (una cotoletta con salsa e riso, un pollo con salsa ai funghi e parmigiano) vengono classificati come `Baked Potato`, probabilmente perché la patata al forno è spesso servita con panna acida o formaggio fuso.

Questa osservazione è coerente con il comportamento di `Baked Potato` sul test set: **recall molto alto (0.98) ma precision più bassa (0.87)**, perché il modello tende ad attribuire a questa classe anche piatti con condimenti simili.

#### Conclusione dell'analisi qualitativa

Su 12 errori osservati, circa **3 sono predizioni ragionevoli** per piatti che appartengono a più categorie, circa **6 riguardano immagini poco rappresentative o con etichetta discutibile** e i restanti derivano da somiglianze visive tra piatti diversi.

L'analisi suggerisce quindi che il F1 Macro non misura solamente la capacità del modello: una parte degli errori dipende dalla **sovrapposizione tra categorie** (un burger di pollo fritto è anche un sandwich) e dalla **qualità del dataset** (confezioni, immagini con testo, etichette assegnate in modo discutibile). In un'applicazione reale questi aspetti potrebbero essere affrontati con una pulizia mirata delle immagini non rappresentative oppure con una classificazione **multi-etichetta**, che permetta di assegnare a un piatto più categorie contemporaneamente.

## 13. Conclusioni

L'obiettivo del progetto era sviluppare e confrontare diverse configurazioni di **EfficientNet pre-addestrata su ImageNet** (B0 e B2) per la classificazione di immagini appartenenti a **14 categorie di alimenti**.

Prima dell'addestramento è stato effettuato un controllo del dataset. Delle **14.000 immagini iniziali**, ne sono state conservate **13.810** dopo la rimozione dei duplicati esatti e dei casi con etichette incompatibili o ambigue individuati durante l'analisi. Il dataset finale utilizzato comprende **8.804 immagini di training, 2.214 di validation e 2.792 di test**.

### Risultati degli esperimenti

Sono state confrontate sette configurazioni controllate, modificando un fattore alla volta tra augmentation, strategia di fine-tuning e tecniche di regolarizzazione, più due esperimenti di approfondimento sulla risoluzione delle immagini e sulla dimensione del modello. Tutti gli esperimenti hanno lo stesso budget massimo di **25 epoche**, con early stopping sul F1 Macro di validation. 

La baseline, nella quale viene allenata solamente la testa finale della rete con augmentation `basic`, ottiene sulla validation un **F1 Macro di 0.8547**; l'augmentation `moderate`, sempre con backbone congelato, lo porta a **0.8654**.

Il miglioramento più importante deriva dal **fine-tuning del backbone**: il fine-tuning in due fasi raggiunge **0.9092** e quello completo fin dall'inizio **0.9128**, circa **4.5 punti** in più rispetto al backbone congelato. La differenza tra le due strategie di fine-tuning (0.36 punti) è invece troppo piccola per indicare una strategia migliore in generale.

Le tecniche di regolarizzazione aggiungono miglioramenti più piccoli ma coerenti: il **Label Smoothing = 0.1** porta il F1 Macro a **0.9194** e rende la validation loss più bassa e stabile, l'augmentation **`strong`** a **0.9229** riducendo il divario tra training e validation. **Mixup (alpha = 0.2)** non produce invece un ulteriore miglioramento (**0.9211**) e aumenta la validation loss: una tecnica di regolarizzazione aggiuntiva non comporta necessariamente prestazioni migliori.

L'esperimento di approfondimento con **immagini 288x288** ottiene il miglior risultato complessivo sulla validation, con **F1 Macro = 0.9281**, migliorando soprattutto il riconoscimento dei dessert, senza aumentare il tempo complessivo di training grazie a una convergenza più rapida. Passare a un modello più grande, **EfficientNet-B2**, non porta un ulteriore miglioramento (**0.9257**): riduce le confusioni tra Taco e Taquito ma peggiora i dessert, segno che il limite principale non è la capacità del modello.

### Modello selezionato

Il modello selezionato esclusivamente sulla base dei risultati di validation è `strong_aug_label_smoothing_288px`, caratterizzato da:

- EfficientNet-B0 pre-addestrata su ImageNet;
- fine-tuning completo del backbone;
- augmentation `strong`;
- Label Smoothing = **0.1**;
- nessun Mixup;
- immagini in input **288x288**.

Sulla validation questa configurazione ha ottenuto **Accuracy = 0.9282**, **F1 Macro = 0.9281** e **Top-3 Accuracy = 0.9783**, con un miglioramento di **7.3 punti** di F1 Macro rispetto alla baseline.

Il modello selezionato è stato successivamente valutato sul **test set**, non utilizzato per scegliere la configurazione finale. Sul test sono stati ottenuti:

| Metrica | Risultato finale |
|---|---:|
| Loss | **0.3917** |
| Accuracy | **0.9108** |
| F1 Macro | **0.9106** |
| F1 Weighted | **0.9105** |
| Top-3 Accuracy | **0.9735** |

Il passaggio dalla validation al test comporta una riduzione di circa **1.75 punti percentuali** sia nell'Accuracy sia nel F1 Macro, mentre la Top-3 Accuracy diminuisce di meno di mezzo punto. Il calo è coerente con il fatto che configurazione e checkpoint sono stati scelti proprio in base alla validation.

### Analisi degli errori

Le prestazioni non sono uniformi tra tutte le categorie. Sul test set la classe più difficile è `Taco`, con un **F1-score di 0.8217** e un recall di 0.7950, seguita da `Cheesecake` (**0.8822**), `Apple Pie` (**0.8861**) e `Taquito` (**0.8934**).

Le confusioni ricorrenti riguardano due aree: la coppia **Taco/Taquito**, a cui si aggiungono gli scambi di `Taco` con `Crispy Chicken`, `Baked Potato` e `Hot Dog`, e i dessert **Apple Pie, Cheesecake e Ice Cream**. La risoluzione più alta ha ridotto le confusioni tra dessert, ma non quella tra Taco e Taquito, che si distinguono soprattutto per la forma della tortilla.

All'inizio dell'analisi, dopo la pulizia del dataset, era stato osservato che la rimozione dei duplicati aveva introdotto un lieve sbilanciamento tra le classi (da 585 a 640 immagini di training) e che sarebbe stato opportuno controllare le **metriche per singola classe**, per verificare che le classi con meno esempi non avessero prestazioni sensibilmente inferiori.

Il controllo, effettuato sul modello selezionato, mostra che **le classi più difficili non dipendono dal numero di immagini**:

- `Fries`, la classe con meno immagini di training (**585**), è tra le classi riconosciute meglio, con un F1 di **0.94** sia in validation sia in test;
- le classi più difficili, `Taco` (**638** immagini), `Apple Pie` e `Cheesecake` (**640**), hanno invece un numero di immagini pari o quasi pari al massimo;
- il F1 medio delle 8 classi ridotte dalla pulizia è molto vicino a quello delle 6 classi rimaste a 640 immagini (**0.925 contro 0.932** in validation, **0.914 contro 0.906** in test).

La differenza massima tra le classi, 55 immagini (meno del 9%), è troppo piccola per influenzare in modo misurabile un modello che parte da feature già apprese su ImageNet. Gli errori dipendono piuttosto dalla somiglianza visiva tra alcuni piatti e dalla qualità di alcune immagini ed etichette; la scelta di non applicare tecniche di riequilibrio, come oversampling o pesi di classe, risulta quindi confermata.

L'osservazione diretta degli errori mostra inoltre che una parte di essi non dipende dal modello: alcuni piatti appartengono a più categorie (un hamburger di pollo fritto è sia `Crispy Chicken` sia `Sandwich`), mentre altre immagini sono confezioni di prodotti, contengono testo sovrapposto o hanno un'etichetta discutibile.

### Limiti del progetto

I risultati devono essere interpretati considerando alcuni limiti:

- gli esperimenti sono stati eseguiti con **un singolo seed**, quindi non è stata misurata la variabilità dei risultati tra più esecuzioni: le differenze inferiori al punto percentuale tra alcune configurazioni non sono sufficienti per stabilire quale sia migliore in generale;
- gli esperimenti con augmentation `moderate` e backbone congelato (esperimento 2) e con Mixup (esperimento 7) hanno raggiunto il miglior risultato all'ultima epoca: con un budget ancora maggiore potrebbero migliorare leggermente;
- alcune immagini sono poco rappresentative della propria classe o hanno un'etichetta discutibile, e alcune categorie si sovrappongono (ad esempio `Crispy Chicken` e `Sandwich`), mentre il dataset assegna a ogni immagine una sola etichetta;
- non è stato introdotto un processo specifico per individuare e correggere sistematicamente il possibile **label noise** presente nell'intero dataset. Una procedura più completa avrebbe richiesto l'identificazione dei campioni sospetti, ad esempio analizzando le immagini sulle quali il modello mostra **bassa confidenza** oppure un **disaccordo persistente tra previsione ed etichetta originale**, seguita da una **revisione manuale** dei casi individuati. Solo dopo tale verifica sarebbe stato possibile confermare l'etichetta originale, correggerla oppure escludere l'immagine dal dataset.

### Considerazioni finali

Nel complesso, gli esperimenti mostrano che il **transfer learning con fine-tuning completo**, combinato con **augmentation strong, Label Smoothing e una risoluzione di input maggiore**, è risultato efficace per questo problema di classificazione.

Il modello finale raggiunge sul test un **F1 Macro di 0.9106** e identifica la classe corretta tra le prime tre predizioni nel **97.35%** dei casi.

I risultati sono quindi positivi sul dataset considerato, ma non sono sufficienti per considerare il sistema direttamente pronto per l'utilizzo su immagini reali esterne al dataset.